# 01 — Runtime Data Inventory and Deterministic Rules Baseline Build

This notebook documents the exploratory and validation work used to build the
deterministic Device Protection Claims Triage baseline.

It covers:

- Runtime data inventory and source validation
- Claim-context assembly and tool-output checks
- Policy, coverage, evidence, risk, and claims-history audits
- Plan-configuration validation
- Deterministic triage-rule design and precedence validation
- Development-claim batch reconciliation
- Baseline decision export creation

This notebook is retained as the detailed build and audit record.

The finalized end-to-end demonstration, deterministic-triage tool validation,
and agent-response guardrail validation are maintained separately in:

`02_deterministic_triage_baseline.ipynb`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to Python path:", PROJECT_ROOT)

Project root added to Python path: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RUNTIME_DIR = PROJECT_ROOT / "data" / "runtime"
KB_DIR = PROJECT_ROOT / "data" / "knowledge_base"

print("Project root:", PROJECT_ROOT)
print("Runtime data folder exists:", RUNTIME_DIR.exists())
print("Knowledge base folder exists:", KB_DIR.exists())

reference_files = sorted((RUNTIME_DIR / "reference").glob("*.csv"))
claim_files = sorted((RUNTIME_DIR / "claims").glob("*.csv"))
kb_files = sorted(KB_DIR.glob("*.md"))

print("\nReference CSV files:", len(reference_files))
for file in reference_files:
    print(" -", file.name)

print("\nClaim CSV files:", len(claim_files))
for file in claim_files:
    print(" -", file.name)

print("\nKnowledge-base documents:", len(kb_files))
for file in kb_files:
    print(" -", file.name)

Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
Runtime data folder exists: True
Knowledge base folder exists: True

Reference CSV files: 30
 - active_version_register_v1_1.csv
 - claim_intake_data_dictionary_v1.csv
 - decision_precedence_v1_1.csv
 - decision_taxonomy_dispositions_v1.csv
 - device_value_reference_v1.csv
 - evidence_metadata_data_dictionary_v1.csv
 - evidence_profile_requirements_v1.csv
 - evidence_type_catalog_v1.csv
 - follow_up_question_catalog_v1.csv
 - follow_up_question_data_dictionary_v1.csv
 - follow_up_question_selection_rules_v1.csv
 - ground_truth_labels_data_dictionary_v1.csv
 - knowledge_base_manifest_data_dictionary_v1.csv
 - knowledge_base_manifest_v1.csv
 - label_source_version_manifest_v1.csv
 - manual_review_reasons_v1.csv
 - plan_coverage_matrix_v1.csv
 - plan_master_data_dictionary_v1.csv
 - policy_eligibility_lookup_data_dictionary_v1.csv
 - policy_eligibility_lookup_v1_1.csv
 - policy_rule_catal

In [3]:
plan_master = pd.read_csv(
    RUNTIME_DIR / "reference" / "protection_plan_master_v1_1.csv"
)

claims_dev = pd.read_csv(
    RUNTIME_DIR / "claims" / "claims_intake_development_v1.csv"
)

print("Plan master shape:", plan_master.shape)
print("Development claims shape:", claims_dev.shape)

display(plan_master.head())
display(claims_dev.head())

Plan master shape: (3, 16)
Development claims shape: (165, 14)


,plan_id,plan_version,plan_name,product_family,plan_tier,operating_market,covered_device_category,policy_duration_months,annual_claim_limit,maximum_theft_claims,default_resolution_strategy,replacement_allowed_if,plan_status,effective_start_date,effective_end_date,plan_description
0,DP-ESSENTIAL,1.1,DeviceProtect Essential,DeviceProtect,ESSENTIAL,US,SMARTPHONE,12,1,0,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,Entry-level protection for accidental screen d...
1,DP-COMPLETE,1.1,DeviceProtect Complete,DeviceProtect,COMPLETE,US,SMARTPHONE,12,2,0,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,"Standard protection for accidental, liquid, an..."
2,DP-PREMIUM,1.1,DeviceProtect Premium,DeviceProtect,PREMIUM,US,SMARTPHONE,12,2,1,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,"Comprehensive protection for accidental, liqui..."


,claim_id,case_created_date,claim_submission_channel,claim_language,policy_id_provided,customer_id_provided,claimed_device_identifier,reported_incident_date,claim_category_selected,requested_service_type,evidence_bundle_id,customer_statement,intake_record_status,record_snapshot_date
0,CLM-000001,2025-07-14,CONTACT_CENTER,EN,POL-00018,CUS-0033,DVC-00000018,2025-07-14,SCREEN_DAMAGE,REPAIR,EBND-000001,I dropped the phone while getting out of my ca...,SUBMITTED,2026-06-23
1,CLM-000002,2024-12-31,WEB_PORTAL,EN,POL-00054,CUS-0013,DVC-00000054,2024-12-23,SCREEN_DAMAGE,UNSPECIFIED,EBND-000002,The phone fell from a table and the front glas...,SUBMITTED,2026-06-23
2,CLM-000003,2025-04-04,WEB_PORTAL,EN,POL-00079,CUS-0003,DVC-00000079,2025-04-01,ACCIDENTAL_DAMAGE,UNSPECIFIED,EBND-000003,I knocked my phone off the counter. The back p...,SUBMITTED,2026-06-23
3,CLM-000004,2024-11-08,WEB_PORTAL,EN,POL-00005,CUS-0006,DVC-00000005,2024-11-03,MECHANICAL_BREAKDOWN,REPAIR,EBND-000004,My phone has started shutting down unexpectedl...,SUBMITTED,2026-06-23
4,CLM-000005,2025-04-30,WEB_PORTAL,EN,POL-00117,CUS-0093,DVC-00000117,2025-04-22,SCREEN_DAMAGE,UNSPECIFIED,EBND-000005,I dropped the phone while getting out of my ca...,SUBMITTED,2026-06-23


In [4]:
print("Plan Master Columns:")
print(plan_master.columns.tolist())

print("\nDevelopment Claims Columns:")
print(claims_dev.columns.tolist())

print("\nPlan Master Preview:")
display(plan_master.head(3))

print("\nDevelopment Claims Preview:")
display(claims_dev.head(3))

Plan Master Columns:
['plan_id', 'plan_version', 'plan_name', 'product_family', 'plan_tier', 'operating_market', 'covered_device_category', 'policy_duration_months', 'annual_claim_limit', 'maximum_theft_claims', 'default_resolution_strategy', 'replacement_allowed_if', 'plan_status', 'effective_start_date', 'effective_end_date', 'plan_description']

Development Claims Columns:
['claim_id', 'case_created_date', 'claim_submission_channel', 'claim_language', 'policy_id_provided', 'customer_id_provided', 'claimed_device_identifier', 'reported_incident_date', 'claim_category_selected', 'requested_service_type', 'evidence_bundle_id', 'customer_statement', 'intake_record_status', 'record_snapshot_date']

Plan Master Preview:


,plan_id,plan_version,plan_name,product_family,plan_tier,operating_market,covered_device_category,policy_duration_months,annual_claim_limit,maximum_theft_claims,default_resolution_strategy,replacement_allowed_if,plan_status,effective_start_date,effective_end_date,plan_description
0,DP-ESSENTIAL,1.1,DeviceProtect Essential,DeviceProtect,ESSENTIAL,US,SMARTPHONE,12,1,0,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,Entry-level protection for accidental screen d...
1,DP-COMPLETE,1.1,DeviceProtect Complete,DeviceProtect,COMPLETE,US,SMARTPHONE,12,2,0,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,"Standard protection for accidental, liquid, an..."
2,DP-PREMIUM,1.1,DeviceProtect Premium,DeviceProtect,PREMIUM,US,SMARTPHONE,12,2,1,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,"Comprehensive protection for accidental, liqui..."



Development Claims Preview:


,claim_id,case_created_date,claim_submission_channel,claim_language,policy_id_provided,customer_id_provided,claimed_device_identifier,reported_incident_date,claim_category_selected,requested_service_type,evidence_bundle_id,customer_statement,intake_record_status,record_snapshot_date
0,CLM-000001,2025-07-14,CONTACT_CENTER,EN,POL-00018,CUS-0033,DVC-00000018,2025-07-14,SCREEN_DAMAGE,REPAIR,EBND-000001,I dropped the phone while getting out of my ca...,SUBMITTED,2026-06-23
1,CLM-000002,2024-12-31,WEB_PORTAL,EN,POL-00054,CUS-0013,DVC-00000054,2024-12-23,SCREEN_DAMAGE,UNSPECIFIED,EBND-000002,The phone fell from a table and the front glas...,SUBMITTED,2026-06-23
2,CLM-000003,2025-04-04,WEB_PORTAL,EN,POL-00079,CUS-0003,DVC-00000079,2025-04-01,ACCIDENTAL_DAMAGE,UNSPECIFIED,EBND-000003,I knocked my phone off the counter. The back p...,SUBMITTED,2026-06-23


In [5]:
print("Development Claims Data Types:")
display(claims_dev.dtypes.to_frame(name="data_type"))

print("\nMissing Values:")
display(
    claims_dev.isna()
    .sum()
    .to_frame(name="missing_count")
    .sort_values("missing_count", ascending=False)
)

print("\nClaim Category Distribution:")
display(
    claims_dev["claim_category_selected"]
    .value_counts(dropna=False)
    .rename_axis("claim_category")
    .reset_index(name="count")
)

print("\nRequested Service Type Distribution:")
display(
    claims_dev["requested_service_type"]
    .value_counts(dropna=False)
    .rename_axis("requested_service_type")
    .reset_index(name="count")
)

Development Claims Data Types:


,data_type
claim_id,str
case_created_date,str
claim_submission_channel,str
claim_language,str
policy_id_provided,str
customer_id_provided,str
claimed_device_identifier,str
reported_incident_date,str
claim_category_selected,str
requested_service_type,str



Missing Values:


,missing_count
claimed_device_identifier,20
policy_id_provided,12
reported_incident_date,11
claim_id,0
case_created_date,0
claim_submission_channel,0
claim_language,0
customer_id_provided,0
claim_category_selected,0
requested_service_type,0



Claim Category Distribution:


,claim_category,count
0,SCREEN_DAMAGE,53
1,MECHANICAL_BREAKDOWN,33
2,ACCIDENTAL_DAMAGE,25
3,LIQUID_DAMAGE,23
4,THEFT,12
5,LOSS,10
6,UNSPECIFIED,9



Requested Service Type Distribution:


,requested_service_type,count
0,UNSPECIFIED,75
1,REPAIR,63
2,REPLACEMENT,27


In [6]:
claims_dev.columns

Index(['claim_id', 'case_created_date', 'claim_submission_channel',
       'claim_language', 'policy_id_provided', 'customer_id_provided',
       'claimed_device_identifier', 'reported_incident_date',
       'claim_category_selected', 'requested_service_type',
       'evidence_bundle_id', 'customer_statement', 'intake_record_status',
       'record_snapshot_date'],
      dtype='str')

In [7]:
sample_claims = (
    claims_dev
    .groupby("claim_category_selected", as_index=False)
    .first()
)

display(
    sample_claims[
        [
            "claim_id",
            "policy_id_provided",
            "customer_id_provided",
            "claimed_device_identifier",
            "reported_incident_date",
            "claim_category_selected",
            "requested_service_type",
            "evidence_bundle_id",
            "customer_statement",
        ]
    ]
)

,claim_id,policy_id_provided,customer_id_provided,claimed_device_identifier,reported_incident_date,claim_category_selected,requested_service_type,evidence_bundle_id,customer_statement
0,CLM-000003,POL-00079,CUS-0003,DVC-00000079,2025-04-01,ACCIDENTAL_DAMAGE,UNSPECIFIED,EBND-000003,I knocked my phone off the counter. The back p...
1,CLM-000011,POL-00061,CUS-0100,DVC-00000061,2026-06-16,LIQUID_DAMAGE,UNSPECIFIED,EBND-000011,My phone was exposed to water and it has not b...
2,CLM-000186,POL-00096,CUS-0098,DVC-00000096,2026-03-15,LOSS,REPLACEMENT,EBND-000186,My device was left behind at a public place an...
3,CLM-000004,POL-00005,CUS-0006,DVC-00000005,2024-11-03,MECHANICAL_BREAKDOWN,REPAIR,EBND-000004,My phone has started shutting down unexpectedl...
4,CLM-000001,POL-00018,CUS-0033,DVC-00000018,2025-07-14,SCREEN_DAMAGE,REPAIR,EBND-000001,I dropped the phone while getting out of my ca...
5,CLM-000015,POL-00015,CUS-0054,DVC-00000015,2026-04-25,THEFT,REPLACEMENT,EBND-000015,My phone was stolen while I was travelling. Pl...
6,CLM-000086,POL-00038,CUS-0007,DVC-00000038,2026-06-05,UNSPECIFIED,UNSPECIFIED,EBND-000086,My registered phone is not working properly af...


In [8]:
policy_lookup = pd.read_csv(
    RUNTIME_DIR / "reference" / "policy_eligibility_lookup_v1_1.csv"
)

print("Policy lookup shape:", policy_lookup.shape)
print("\nColumns:")
print(policy_lookup.columns.tolist())

display(policy_lookup.head(3))

Policy lookup shape: (120, 18)

Columns:
['customer_id', 'policy_id', 'plan_id', 'policy_status', 'policy_enrollment_date', 'coverage_start_date', 'coverage_end_date', 'policy_termination_date', 'suspension_start_date', 'suspension_end_date', 'covered_device_id', 'covered_device_identifier', 'device_category', 'device_brand', 'device_model', 'device_purchase_date', 'device_enrollment_date', 'record_snapshot_date']


,customer_id,policy_id,plan_id,policy_status,policy_enrollment_date,coverage_start_date,coverage_end_date,policy_termination_date,suspension_start_date,suspension_end_date,covered_device_id,covered_device_identifier,device_category,device_brand,device_model,device_purchase_date,device_enrollment_date,record_snapshot_date
0,CUS-0055,POL-00001,DP-COMPLETE,ACTIVE,2026-06-10,2026-06-10,2027-06-09,NaN,NaN,NaN,CDV-00001,DVC-00000001,SMARTPHONE,Cobalt,Cobalt Mini,2026-04-20,2026-06-10,2026-06-23
1,CUS-0053,POL-00002,DP-COMPLETE,EXPIRED,2024-07-03,2024-07-03,2025-07-02,NaN,NaN,NaN,CDV-00002,DVC-00000002,SMARTPHONE,Orion,X5,2024-05-08,2024-07-03,2026-06-23
2,CUS-0064,POL-00003,DP-ESSENTIAL,EXPIRED,2024-10-07,2024-10-07,2025-10-06,NaN,NaN,NaN,CDV-00003,DVC-00000003,SMARTPHONE,Lumen,Lumen Pro,2024-02-20,2024-10-07,2026-06-23


In [9]:
prior_claims = pd.read_csv(
    RUNTIME_DIR / "reference" / "prior_claims_history_v1_2.csv"
)

print("Prior claims history shape:", prior_claims.shape)
print("\nColumns:")
print(prior_claims.columns.tolist())

display(prior_claims.head(5))

Prior claims history shape: (112, 16)

Columns:
['historical_claim_id', 'policy_id', 'customer_id', 'covered_device_id', 'plan_id', 'incident_date', 'claim_reported_date', 'claim_category', 'final_claim_status', 'resolution_type', 'closure_reason_code', 'claim_limit_consumed_flag', 'theft_limit_consumed_flag', 'settled_amount_usd', 'currency', 'record_snapshot_date']


,historical_claim_id,policy_id,customer_id,covered_device_id,plan_id,incident_date,claim_reported_date,claim_category,final_claim_status,resolution_type,closure_reason_code,claim_limit_consumed_flag,theft_limit_consumed_flag,settled_amount_usd,currency,record_snapshot_date
0,HCLM-000001,POL-00003,CUS-0064,CDV-00003,DP-ESSENTIAL,2024-10-14,2024-11-01,SCREEN_DAMAGE,SETTLED,REPAIR,COVERED_CLAIM_SETTLED,True,False,119.0,USD,2026-06-23
1,HCLM-000002,POL-00003,CUS-0064,CDV-00003,DP-ESSENTIAL,2025-03-27,2025-03-31,ACCIDENTAL_DAMAGE,DECLINED,NONE,PLAN_EXCLUSION,False,False,NaN,NaN,2026-06-23
2,HCLM-000003,POL-00003,CUS-0064,CDV-00003,DP-ESSENTIAL,2025-05-11,2025-05-16,SCREEN_DAMAGE,DECLINED,NONE,CLAIM_LIMIT_EXHAUSTED,False,False,NaN,NaN,2026-06-23
3,HCLM-000004,POL-00004,CUS-0019,CDV-00004,DP-ESSENTIAL,2025-11-12,2025-11-18,LIQUID_DAMAGE,DECLINED,NONE,PLAN_EXCLUSION,False,False,NaN,NaN,2026-06-23
4,HCLM-000005,POL-00005,CUS-0006,CDV-00005,DP-COMPLETE,2025-02-03,2025-02-08,SCREEN_DAMAGE,SETTLED,REPLACEMENT,COVERED_CLAIM_SETTLED,True,False,999.0,USD,2026-06-23


In [10]:
evidence_bundles = pd.read_csv(
    RUNTIME_DIR / "claims" / "evidence_bundle_manifest_v1.csv"
)

evidence_docs = pd.read_csv(
    RUNTIME_DIR / "claims" / "claim_evidence_document_metadata_v1.csv"
)

print("Evidence bundle shape:", evidence_bundles.shape)
print("Evidence document metadata shape:", evidence_docs.shape)

print("\nEvidence bundle columns:")
print(evidence_bundles.columns.tolist())

print("\nEvidence document metadata columns:")
print(evidence_docs.columns.tolist())

display(evidence_bundles.head(3))
display(evidence_docs.head(5))

Evidence bundle shape: (220, 8)
Evidence document metadata shape: (283, 14)

Evidence bundle columns:
['evidence_bundle_id', 'claim_id', 'evidence_submission_status', 'received_document_count', 'first_received_date', 'last_received_date', 'evidence_source_system', 'record_snapshot_date']

Evidence document metadata columns:
['evidence_item_id', 'evidence_bundle_id', 'claim_id', 'evidence_type', 'document_status', 'submitted_date', 'document_reference', 'extracted_incident_date', 'extracted_claim_category', 'extracted_device_identifier', 'repair_estimate_usd', 'police_report_reference', 'document_summary', 'record_snapshot_date']


,evidence_bundle_id,claim_id,evidence_submission_status,received_document_count,first_received_date,last_received_date,evidence_source_system,record_snapshot_date
0,EBND-000001,CLM-000001,DOCUMENTS_SUBMITTED,2,2025-07-17,2025-07-17,AGENT_CASE_ATTACHMENTS,2026-06-23
1,EBND-000002,CLM-000002,DOCUMENTS_SUBMITTED,2,2024-12-23,2024-12-23,CUSTOMER_PORTAL,2026-06-23
2,EBND-000003,CLM-000003,DOCUMENTS_SUBMITTED,1,2025-04-03,2025-04-03,CUSTOMER_PORTAL,2026-06-23


,evidence_item_id,evidence_bundle_id,claim_id,evidence_type,document_status,submitted_date,document_reference,extracted_incident_date,extracted_claim_category,extracted_device_identifier,repair_estimate_usd,police_report_reference,document_summary,record_snapshot_date
0,EVI-000001,EBND-000001,CLM-000001,DAMAGE_PHOTO,RECEIVED_VALID,2025-07-17,IMG-000001,2025-07-14,SCREEN_DAMAGE,DVC-00000018,NaN,NaN,Upload summary: Cracked display and touch-func...,2026-06-23
1,EVI-000002,EBND-000001,CLM-000001,REPAIR_QUOTE,RECEIVED_VALID,2025-07-17,RQT-000002,2025-07-14,SCREEN_DAMAGE,DVC-00000018,219.0,NaN,Repair provider estimate dated 2025-07-17.,2026-06-23
2,EVI-000003,EBND-000002,CLM-000002,DAMAGE_PHOTO,RECEIVED_VALID,2024-12-23,IMG-000003,2024-12-23,SCREEN_DAMAGE,DVC-00000054,NaN,NaN,Upload summary: Cracked display and touch-func...,2026-06-23
3,EVI-000004,EBND-000002,CLM-000002,REPAIR_QUOTE,RECEIVED_VALID,2024-12-23,RQT-000004,2024-12-23,SCREEN_DAMAGE,DVC-00000054,149.0,NaN,Repair provider estimate dated 2024-12-23.,2026-06-23
4,EVI-000005,EBND-000003,CLM-000003,DAMAGE_PHOTO,RECEIVED_VALID,2025-04-03,IMG-000005,2025-04-01,ACCIDENTAL_DAMAGE,DVC-00000079,NaN,NaN,Upload summary: Physical impact damage is visi...,2026-06-23


In [11]:
policy_rules = pd.read_csv(
    RUNTIME_DIR / "reference" / "policy_rule_catalog_v1_2.csv"
)

print("Policy rule catalogue shape:", policy_rules.shape)
print("\nColumns:")
print(policy_rules.columns.tolist())

display(
    policy_rules[
        [
            "rule_id",
            "rule_name",
            "rule_domain",
            "priority",
            "required_input_fields",
            "outcome_on_failure",
            "manual_review_reason",
            "severity",
            "active_flag",
        ]
    ]
    .sort_values("priority")
)

Policy rule catalogue shape: (19, 20)

Columns:
['rule_id', 'rule_name', 'rule_domain', 'priority', 'applies_to_plan_id', 'applies_to_claim_category', 'applies_when', 'required_input_fields', 'pass_condition', 'failure_condition', 'missing_or_unknown_handling', 'outcome_on_failure', 'manual_review_reason', 'policy_reference', 'severity', 'rationale', 'rule_version', 'effective_start_date', 'effective_end_date', 'active_flag']


,rule_id,rule_name,rule_domain,priority,required_input_fields,outcome_on_failure,manual_review_reason,severity,active_flag
0,DATA-001,Unique policy eligibility record available,Data Validation,10,customer_id OR policy_id; policy eligibility l...,INFO_REQUIRED,NaN,BLOCKING,Yes
1,DATA-002,Conflicting or multiple policy records,Data Validation,10,policy eligibility lookup result; source-syste...,MANUAL_REVIEW,DATA_CONFLICT,BLOCKING,Yes
2,DATA-003,Plan version and coverage configuration available,Data Validation,10,plan_id; plan version/effective dates; plan co...,MANUAL_REVIEW,RULE_NOT_AVAILABLE,BLOCKING,Yes
14,EXC-002,Suspected or unclear exclusion,Exclusion,10,exclusion_status; exclusion_type; narrative_co...,MANUAL_REVIEW,POLICY_EXCEPTION,BLOCKING,Yes
5,ELG-003,Supported product and operating scope,Eligibility,10,product_family; device_category; operating_reg...,MANUAL_REVIEW,RULE_NOT_AVAILABLE,BLOCKING,Yes
7,DEV-002,Unresolved claimed-device mismatch,Device Validation,10,claimed_device_identifier; enrolled_device_ide...,MANUAL_REVIEW,DEVICE_MISMATCH,BLOCKING,Yes
17,EVD-002,Evidence internally consistent and authenticat...,Evidence,10,evidence_item_statuses; evidence_consistency_s...,MANUAL_REVIEW,DATA_CONFLICT,BLOCKING,Yes
4,ELG-002,Policy active on incident date,Eligibility,20,coverage_start_date; coverage_end_date; policy...,NOT_ELIGIBLE,NaN,BLOCKING,Yes
13,EXC-001,Verified global exclusion does not apply,Exclusion,20,exclusion_status; exclusion_type; verified_evi...,NOT_ELIGIBLE,NaN,BLOCKING,Yes
8,DEV-003,Verified device not enrolled,Device Validation,20,claimed_device_identifier; enrolled_device_ide...,NOT_ELIGIBLE,NaN,BLOCKING,Yes


In [12]:
coverage_matrix = pd.read_csv(
    RUNTIME_DIR / "reference" / "plan_coverage_matrix_v1.csv"
)

print("Coverage matrix shape:", coverage_matrix.shape)
print("\nColumns:")
print(coverage_matrix.columns.tolist())

display(
    coverage_matrix[
        [
            "plan_id",
            "claim_category",
            "covered_flag",
            "deductible_amount",
            "repair_eligible_flag",
            "replacement_eligible_flag",
            "evidence_profile_id",
        ]
    ]
    .sort_values(["plan_id", "claim_category"])
)

Coverage matrix shape: (18, 11)

Columns:
['coverage_matrix_id', 'plan_id', 'claim_category', 'category_scope', 'covered_flag', 'deductible_amount', 'currency', 'evidence_profile_id', 'repair_eligible_flag', 'replacement_eligible_flag', 'coverage_notes']


,plan_id,claim_category,covered_flag,deductible_amount,repair_eligible_flag,replacement_eligible_flag,evidence_profile_id
7,DP-COMPLETE,ACCIDENTAL_DAMAGE,True,49.0,True,True,EVD-ACCIDENTAL-01
8,DP-COMPLETE,LIQUID_DAMAGE,True,49.0,True,True,EVD-LIQUID-01
11,DP-COMPLETE,LOSS,False,NaN,False,False,NaN
9,DP-COMPLETE,MECHANICAL_BREAKDOWN,True,0.0,True,True,EVD-MECHANICAL-01
6,DP-COMPLETE,SCREEN_DAMAGE,True,49.0,True,True,EVD-SCREEN-01
10,DP-COMPLETE,THEFT,False,NaN,False,False,NaN
1,DP-ESSENTIAL,ACCIDENTAL_DAMAGE,False,NaN,False,False,NaN
2,DP-ESSENTIAL,LIQUID_DAMAGE,False,NaN,False,False,NaN
5,DP-ESSENTIAL,LOSS,False,NaN,False,False,NaN
3,DP-ESSENTIAL,MECHANICAL_BREAKDOWN,False,NaN,False,False,NaN


In [13]:
from src.data_loader import load_runtime_data

data = load_runtime_data()

print("Loaded datasets:")
for name, df in data.items():
    print(f"{name}: {df.shape}")

Loaded datasets:
plan_master: (3, 16)
coverage_matrix: (18, 11)
evidence_requirements: (9, 7)
policy_rules: (19, 20)
policy_lookup: (120, 18)
prior_claims: (112, 16)
evidence_bundles: (220, 8)
evidence_documents: (283, 14)
development_claims: (165, 14)
risk_results: (4, 19)


In [14]:
from pathlib import Path

print("Current notebook folder:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("src folder exists:", (PROJECT_ROOT / "src").exists())
print("data_validation.py exists:", (PROJECT_ROOT / "src" / "data_validation.py").exists())

print("\nFiles inside src:")
for file in sorted((PROJECT_ROOT / "src").iterdir()):
    print("-", file.name)

Current notebook folder: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/notebooks
Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
src folder exists: True
data_validation.py exists: True

Files inside src:
- __init__.py
- __pycache__
- claim_context.py
- data_loader.py
- data_validation.py
- tools
- triage_engine.py


In [15]:
from src.data_validation import (
    validate_required_columns,
    validate_basic_record_counts,
)

column_results = validate_required_columns(data)
count_results = validate_basic_record_counts(data)

print("Required-column validation")
for result in column_results:
    print(result)

print("\nBasic record-count validation")
for result in count_results:
    print(result)

Required-column validation
PASS - plan_master: required columns available (3 checked)
PASS - coverage_matrix: required columns available (4 checked)
PASS - policy_lookup: required columns available (6 checked)
PASS - prior_claims: required columns available (5 checked)
PASS - development_claims: required columns available (6 checked)

Basic record-count validation
PASS - plan_master: 3 rows
PASS - coverage_matrix: 18 rows
PASS - policy_lookup: 120 rows
PASS - prior_claims: 112 rows
PASS - development_claims: 165 rows


In [16]:
from src.tools.policy_lookup import find_policy_records

sample_claim = data["development_claims"].iloc[0]

matches = find_policy_records(
    policy_lookup=data["policy_lookup"],
    policy_id=sample_claim["policy_id_provided"],
    customer_id=sample_claim["customer_id_provided"],
    device_identifier=sample_claim["claimed_device_identifier"],
)

print("Selected claim:", sample_claim["claim_id"])
print("Policy ID provided:", sample_claim["policy_id_provided"])
print("Customer ID provided:", sample_claim["customer_id_provided"])
print("Device identifier provided:", sample_claim["claimed_device_identifier"])
print("Matching policy records:", len(matches))

display(
    matches[
        [
            "customer_id",
            "policy_id",
            "plan_id",
            "policy_status",
            "covered_device_identifier",
            "coverage_start_date",
            "coverage_end_date",
        ]
    ]
)

Selected claim: CLM-000001
Policy ID provided: POL-00018
Customer ID provided: CUS-0033
Device identifier provided: DVC-00000018
Matching policy records: 1


,customer_id,policy_id,plan_id,policy_status,covered_device_identifier,coverage_start_date,coverage_end_date
17,CUS-0033,POL-00018,DP-ESSENTIAL,EXPIRED,DVC-00000018,2024-12-25,2025-12-24


In [17]:
from src.tools.policy_lookup import find_policy_records

test_cases = [
    {
        "scenario": "Policy ID lookup",
        "policy_id": "POL-00018",
        "customer_id": None,
        "device_identifier": None,
    },
    {
        "scenario": "Customer + device lookup",
        "policy_id": None,
        "customer_id": "CUS-0033",
        "device_identifier": "DVC-00000018",
    },
    {
        "scenario": "Customer-only lookup",
        "policy_id": None,
        "customer_id": "CUS-0033",
        "device_identifier": None,
    },
    {
        "scenario": "No identifiers provided",
        "policy_id": None,
        "customer_id": None,
        "device_identifier": None,
    },
]

for case in test_cases:
    matches = find_policy_records(
        policy_lookup=data["policy_lookup"],
        policy_id=case["policy_id"],
        customer_id=case["customer_id"],
        device_identifier=case["device_identifier"],
    )

    print(f"\nScenario: {case['scenario']}")
    print("Matching records:", len(matches))

    if not matches.empty:
        display(
            matches[
                [
                    "customer_id",
                    "policy_id",
                    "plan_id",
                    "policy_status",
                    "covered_device_identifier",
                ]
            ]
        )


Scenario: Policy ID lookup
Matching records: 1


,customer_id,policy_id,plan_id,policy_status,covered_device_identifier
17,CUS-0033,POL-00018,DP-ESSENTIAL,EXPIRED,DVC-00000018



Scenario: Customer + device lookup
Matching records: 1


,customer_id,policy_id,plan_id,policy_status,covered_device_identifier
17,CUS-0033,POL-00018,DP-ESSENTIAL,EXPIRED,DVC-00000018



Scenario: Customer-only lookup
Matching records: 2


,customer_id,policy_id,plan_id,policy_status,covered_device_identifier
17,CUS-0033,POL-00018,DP-ESSENTIAL,EXPIRED,DVC-00000018
38,CUS-0033,POL-00039,DP-PREMIUM,ACTIVE,DVC-00000039



Scenario: No identifiers provided
Matching records: 0


In [18]:
from src.tools.policy_lookup import classify_lookup_result

for case in test_cases:
    matches = find_policy_records(
        policy_lookup=data["policy_lookup"],
        policy_id=case["policy_id"],
        customer_id=case["customer_id"],
        device_identifier=case["device_identifier"],
    )

    print(
        f"{case['scenario']}: "
        f"{classify_lookup_result(matches)} "
        f"({len(matches)} record(s))"
    )

Policy ID lookup: UNIQUE_MATCH (1 record(s))
Customer + device lookup: UNIQUE_MATCH (1 record(s))
Customer-only lookup: MULTIPLE_MATCH (2 record(s))
No identifiers provided: NO_MATCH (0 record(s))


In [19]:
from src.tools.claims_history import (
    get_policy_claim_history,
    calculate_claim_usage,
)

sample_claim = data["development_claims"].iloc[0]

policy_id = sample_claim["policy_id_provided"]
incident_date = sample_claim["reported_incident_date"]

history = get_policy_claim_history(
    prior_claims=data["prior_claims"],
    policy_id=policy_id,
    incident_date=incident_date,
)

usage = calculate_claim_usage(history)

print("Claim ID:", sample_claim["claim_id"])
print("Policy ID:", policy_id)
print("Current incident date:", incident_date)
print("Historical claims before current incident:", len(history))
print("Usage summary:", usage)

display(
    history[
        [
            "historical_claim_id",
            "incident_date",
            "claim_category",
            "final_claim_status",
            "claim_limit_consumed_flag",
            "theft_limit_consumed_flag",
        ]
    ]
    .sort_values("incident_date")
)

Claim ID: CLM-000001
Policy ID: POL-00018
Current incident date: 2025-07-14
Historical claims before current incident: 0
Usage summary: {'annual_claims_used': 0, 'theft_claims_used': 0}


,historical_claim_id,incident_date,claim_category,final_claim_status,claim_limit_consumed_flag,theft_limit_consumed_flag


In [20]:
history_usage_by_policy = (
    data["prior_claims"]
    .assign(
        annual_consumed=lambda df: (
            df["claim_limit_consumed_flag"]
            .astype(str)
            .str.upper()
            .eq("TRUE")
            .astype(int)
        ),
        theft_consumed=lambda df: (
            df["theft_limit_consumed_flag"]
            .astype(str)
            .str.upper()
            .eq("TRUE")
            .astype(int)
        ),
    )
    .groupby("policy_id", as_index=False)
    .agg(
        historical_claim_count=("historical_claim_id", "count"),
        annual_claims_used=("annual_consumed", "sum"),
        theft_claims_used=("theft_consumed", "sum"),
    )
    .sort_values(
        ["annual_claims_used", "theft_claims_used", "historical_claim_count"],
        ascending=False,
    )
)

display(history_usage_by_policy.head(10))

,policy_id,historical_claim_count,annual_claims_used,theft_claims_used
68,POL-00119,4,2,0
11,POL-00023,3,2,0
27,POL-00049,3,2,0
57,POL-00095,3,2,0
58,POL-00096,3,2,0
3,POL-00007,2,2,0
26,POL-00048,2,2,0
34,POL-00060,2,2,0
38,POL-00064,2,2,0
62,POL-00105,2,1,1


In [21]:
test_policy_id = history_usage_by_policy.iloc[0]["policy_id"]

history = get_policy_claim_history(
    prior_claims=data["prior_claims"],
    policy_id=test_policy_id,
)

usage = calculate_claim_usage(history)

print("Test policy ID:", test_policy_id)
print("Historical claims found:", len(history))
print("Usage summary:", usage)

display(
    history[
        [
            "historical_claim_id",
            "incident_date",
            "claim_category",
            "final_claim_status",
            "claim_limit_consumed_flag",
            "theft_limit_consumed_flag",
        ]
    ]
    .sort_values("incident_date")
)

Test policy ID: POL-00119
Historical claims found: 4
Usage summary: {'annual_claims_used': 2, 'theft_claims_used': 0}


,historical_claim_id,incident_date,claim_category,final_claim_status,claim_limit_consumed_flag,theft_limit_consumed_flag
108,HCLM-000109,2025-09-21,ACCIDENTAL_DAMAGE,SETTLED,True,False
107,HCLM-000108,2025-10-04,SCREEN_DAMAGE,SETTLED,True,False
109,HCLM-000110,2026-04-29,LOSS,DECLINED,False,False
110,HCLM-000111,2026-05-09,SCREEN_DAMAGE,DECLINED,False,False


In [22]:
from src.tools.evidence_lookup import (
    get_evidence_bundle,
    get_evidence_documents,
)

sample_claim = data["development_claims"].iloc[0]

bundle = get_evidence_bundle(
    evidence_bundles=data["evidence_bundles"],
    evidence_bundle_id=sample_claim["evidence_bundle_id"],
)

documents = get_evidence_documents(
    evidence_documents=data["evidence_documents"],
    evidence_bundle_id=sample_claim["evidence_bundle_id"],
)

print("Claim ID:", sample_claim["claim_id"])
print("Evidence bundle ID:", sample_claim["evidence_bundle_id"])
print("Bundle records found:", len(bundle))
print("Document records found:", len(documents))

display(bundle)

display(
    documents[
        [
            "evidence_item_id",
            "evidence_type",
            "document_status",
            "submitted_date",
            "extracted_incident_date",
            "extracted_claim_category",
            "extracted_device_identifier",
            "repair_estimate_usd",
            "police_report_reference",
            "document_summary",
        ]
    ]
)

Claim ID: CLM-000001
Evidence bundle ID: EBND-000001
Bundle records found: 1
Document records found: 2


,evidence_bundle_id,claim_id,evidence_submission_status,received_document_count,first_received_date,last_received_date,evidence_source_system,record_snapshot_date
0,EBND-000001,CLM-000001,DOCUMENTS_SUBMITTED,2,2025-07-17,2025-07-17,AGENT_CASE_ATTACHMENTS,2026-06-23


,evidence_item_id,evidence_type,document_status,submitted_date,extracted_incident_date,extracted_claim_category,extracted_device_identifier,repair_estimate_usd,police_report_reference,document_summary
0,EVI-000001,DAMAGE_PHOTO,RECEIVED_VALID,2025-07-17,2025-07-14,SCREEN_DAMAGE,DVC-00000018,NaN,NaN,Upload summary: Cracked display and touch-func...
1,EVI-000002,REPAIR_QUOTE,RECEIVED_VALID,2025-07-17,2025-07-14,SCREEN_DAMAGE,DVC-00000018,219.0,NaN,Repair provider estimate dated 2025-07-17.


In [23]:
print("Evidence bundle submission-status distribution:")
display(
    data["evidence_bundles"]["evidence_submission_status"]
    .value_counts(dropna=False)
    .rename_axis("evidence_submission_status")
    .reset_index(name="count")
)

print("\nDocument-status distribution:")
display(
    data["evidence_documents"]["document_status"]
    .value_counts(dropna=False)
    .rename_axis("document_status")
    .reset_index(name="count")
)

print("\nEvidence-type distribution:")
display(
    data["evidence_documents"]["evidence_type"]
    .value_counts(dropna=False)
    .rename_axis("evidence_type")
    .reset_index(name="count")
)

Evidence bundle submission-status distribution:


,evidence_submission_status,count
0,DOCUMENTS_SUBMITTED,193
1,NO_DOCUMENTS_SUBMITTED,27



Document-status distribution:


,document_status,count
0,RECEIVED_VALID,260
1,RECEIVED_CONTRADICTORY,18
2,RECEIVED_UNREADABLE,5



Evidence-type distribution:


,evidence_type,count
0,DAMAGE_PHOTO,146
1,REPAIR_QUOTE,85
2,DIAGNOSTIC_REPORT,39
3,POLICE_REPORT_REFERENCE,13


In [24]:
evidence_requirements = pd.read_csv(
    RUNTIME_DIR / "reference" / "evidence_profile_requirements_v1.csv"
)

print("Evidence requirements shape:", evidence_requirements.shape)
print("\nColumns:")
print(evidence_requirements.columns.tolist())

display(
    evidence_requirements.sort_values(
        ["evidence_profile_id", "evidence_type"]
    )
)

Evidence requirements shape: (9, 7)

Columns:
['evidence_profile_id', 'requirement_sequence', 'evidence_type', 'requirement_level', 'acceptance_criteria', 'handling_if_missing_or_invalid', 'business_purpose']


,evidence_profile_id,requirement_sequence,evidence_type,requirement_level,acceptance_criteria,handling_if_missing_or_invalid,business_purpose
2,EVD-ACCIDENTAL-01,1,DAMAGE_PHOTO,REQUIRED,At least one readable image or upload summary ...,INFO_REQUIRED,Confirm visible accidental damage.
3,EVD-ACCIDENTAL-01,2,REPAIR_QUOTE,OPTIONAL,Readable repair estimate from a repair provide...,NOT_BLOCKING,Support repair feasibility/cost assessment.
4,EVD-LIQUID-01,1,DAMAGE_PHOTO,REQUIRED,At least one readable image or upload summary ...,INFO_REQUIRED,Confirm observed liquid-damage symptoms.
5,EVD-LIQUID-01,2,REPAIR_QUOTE,OPTIONAL,Readable repair estimate from a repair provide...,NOT_BLOCKING,Support repair feasibility/cost assessment.
6,EVD-MECHANICAL-01,1,DIAGNOSTIC_REPORT,REQUIRED,Readable diagnostic report or provider assessm...,INFO_REQUIRED,Confirm fault under normal-use assessment.
7,EVD-MECHANICAL-01,2,REPAIR_QUOTE,OPTIONAL,Readable repair estimate from a repair provide...,NOT_BLOCKING,Support repair feasibility/cost assessment.
0,EVD-SCREEN-01,1,DAMAGE_PHOTO,REQUIRED,At least one readable image or upload summary ...,INFO_REQUIRED,Confirm visible screen damage.
1,EVD-SCREEN-01,2,REPAIR_QUOTE,OPTIONAL,Readable repair estimate from a repair provide...,NOT_BLOCKING,Support repair feasibility/cost assessment.
8,EVD-THEFT-01,1,POLICE_REPORT_REFERENCE,REQUIRED,Readable police-report reference that identifi...,INFO_REQUIRED,Provide theft incident reference for verificat...


In [25]:
from src.tools.evidence_lookup import get_evidence_documents
from src.tools.evidence_assessment import assess_evidence

sample_claim = data["development_claims"].iloc[0]

documents = get_evidence_documents(
    evidence_documents=data["evidence_documents"],
    evidence_bundle_id=sample_claim["evidence_bundle_id"],
)

assessment = assess_evidence(
    evidence_requirements=evidence_requirements,
    evidence_documents=documents,
    evidence_profile_id="EVD-SCREEN-01",
)

print("Claim ID:", sample_claim["claim_id"])
print("Evidence profile:", assessment["evidence_profile_id"])
print("Required evidence:", assessment["required_evidence_types"])
print("Submitted evidence:", assessment["submitted_evidence_types"])
print("Missing required evidence:", assessment["missing_required_evidence_types"])
print("Unreadable required evidence:", assessment["unreadable_required_evidence_types"])
print("Contradictory evidence:", assessment["has_contradictory_evidence"])
print("Assessment status:", assessment["evidence_assessment_status"])

Claim ID: CLM-000001
Evidence profile: EVD-SCREEN-01
Required evidence: ['DAMAGE_PHOTO']
Submitted evidence: ['DAMAGE_PHOTO', 'REPAIR_QUOTE']
Missing required evidence: []
Unreadable required evidence: []
Contradictory evidence: False
Assessment status: SUFFICIENT


In [26]:
# Find one claim with no submitted documents
no_docs_bundle = (
    data["evidence_bundles"]
    .query("evidence_submission_status == 'NO_DOCUMENTS_SUBMITTED'")
    .iloc[0]
)

no_docs_claim_id = no_docs_bundle["claim_id"]
no_docs_claim = data["development_claims"].query(
    "claim_id == @no_docs_claim_id"
).iloc[0]

no_docs = get_evidence_documents(
    evidence_documents=data["evidence_documents"],
    evidence_bundle_id=no_docs_bundle["evidence_bundle_id"],
)

print("No-document claim:", no_docs_claim_id)
print("Selected category:", no_docs_claim["claim_category_selected"])
print("Documents found:", len(no_docs))

No-document claim: CLM-000086
Selected category: UNSPECIFIED
Documents found: 0


In [27]:
# Find one bundle containing contradictory evidence
contradictory_doc = (
    data["evidence_documents"]
    .query("document_status == 'RECEIVED_CONTRADICTORY'")
    .iloc[0]
)

contradictory_docs = get_evidence_documents(
    evidence_documents=data["evidence_documents"],
    evidence_bundle_id=contradictory_doc["evidence_bundle_id"],
)

print("Contradictory claim:", contradictory_doc["claim_id"])
print("Evidence bundle:", contradictory_doc["evidence_bundle_id"])
display(
    contradictory_docs[
        [
            "evidence_type",
            "document_status",
            "extracted_incident_date",
            "extracted_claim_category",
            "extracted_device_identifier",
            "document_summary",
        ]
    ]
)

Contradictory claim: CLM-000144
Evidence bundle: EBND-000144


,evidence_type,document_status,extracted_incident_date,extracted_claim_category,extracted_device_identifier,document_summary
134,POLICE_REPORT_REFERENCE,RECEIVED_CONTRADICTORY,2026-05-29,THEFT,DVC-00000044,Document date indicates the incident occurred ...


In [28]:
contradictory_claim_id = "CLM-000144"

contradictory_claim = data["development_claims"].query(
    "claim_id == @contradictory_claim_id"
).iloc[0]

contradictory_documents = get_evidence_documents(
    evidence_documents=data["evidence_documents"],
    evidence_bundle_id=contradictory_claim["evidence_bundle_id"],
)

contradictory_assessment = assess_evidence(
    evidence_requirements=evidence_requirements,
    evidence_documents=contradictory_documents,
    evidence_profile_id="EVD-THEFT-01",
)

print("Claim ID:", contradictory_claim_id)
print("Selected category:", contradictory_claim["claim_category_selected"])
print("Required evidence:", contradictory_assessment["required_evidence_types"])
print("Submitted evidence:", contradictory_assessment["submitted_evidence_types"])
print("Missing required evidence:", contradictory_assessment["missing_required_evidence_types"])
print("Unreadable required evidence:", contradictory_assessment["unreadable_required_evidence_types"])
print("Contradictory evidence:", contradictory_assessment["has_contradictory_evidence"])
print("Assessment status:", contradictory_assessment["evidence_assessment_status"])

Claim ID: CLM-000144
Selected category: THEFT
Required evidence: ['POLICE_REPORT_REFERENCE']
Submitted evidence: ['POLICE_REPORT_REFERENCE']
Missing required evidence: []
Unreadable required evidence: []
Contradictory evidence: True
Assessment status: CONTRADICTORY


In [29]:
from src.tools.coverage_lookup import (
    get_coverage_record,
    classify_coverage_result,
)

coverage_record = get_coverage_record(
    coverage_matrix=data["coverage_matrix"],
    plan_id="DP-PREMIUM",
    claim_category="THEFT",
)

coverage_result = classify_coverage_result(coverage_record)

print("Coverage result:", coverage_result)
display(coverage_record)

Coverage result: {'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD', 'covered_flag': True, 'evidence_profile_id': 'EVD-THEFT-01'}


,coverage_matrix_id,plan_id,claim_category,category_scope,covered_flag,deductible_amount,currency,evidence_profile_id,repair_eligible_flag,replacement_eligible_flag,coverage_notes
16,DP-PREMIUM-THEFT,DP-PREMIUM,THEFT,SUPPORTED,True,99.0,USD,EVD-THEFT-01,False,True,Covered theft claim. A replacement is the expe...


In [30]:
uncovered_record = get_coverage_record(
    coverage_matrix=data["coverage_matrix"],
    plan_id="DP-ESSENTIAL",
    claim_category="THEFT",
)

uncovered_result = classify_coverage_result(uncovered_record)

print("Uncovered coverage result:", uncovered_result)
display(uncovered_record)

Uncovered coverage result: {'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD', 'covered_flag': False, 'evidence_profile_id': None}


,coverage_matrix_id,plan_id,claim_category,category_scope,covered_flag,deductible_amount,currency,evidence_profile_id,repair_eligible_flag,replacement_eligible_flag,coverage_notes
4,DP-ESSENTIAL-THEFT,DP-ESSENTIAL,THEFT,SUPPORTED,False,NaN,NaN,NaN,False,False,Not covered under DeviceProtect Essential.


In [31]:
from pprint import pprint

from src.claim_context import assemble_claim_context

context = assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

pprint(context)

{'claim': {'claim_category_selected': 'SCREEN_DAMAGE',
           'claim_id': 'CLM-000001',
           'claimed_device_identifier': 'DVC-00000018',
           'customer_id_provided': 'CUS-0033',
           'evidence_bundle_id': 'EBND-000001',
           'policy_id_provided': 'POL-00018',
           'reported_incident_date': '2025-07-14',
           'requested_service_type': 'REPAIR'},
 'claim_history': {'annual_claims_used': 0,
                   'historical_claim_count': 0,
                   'theft_claims_used': 0},
 'coverage': {'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD',
              'covered_flag': True,
              'evidence_profile_id': 'EVD-SCREEN-01'},
 'device_match': {'claimed_device_identifier': 'DVC-00000018',
                  'covered_device_identifier': 'DVC-00000018',
                  'status': 'DEVICE_MATCH'},
 'evidence': {'evidence_assessment_status': 'SUFFICIENT',
              'evidence_profile_id': 'EVD-SCREEN-01',
              'has_contradictory_evi

In [32]:
context = assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

pprint(context["policy_eligibility"])

{'policy_active_for_incident': True,
 'policy_incident_status': 'ACTIVE_ON_INCIDENT_DATE',
 'reason': 'Incident date falls within the active policy coverage period.'}


In [33]:
contexts = []

for claim_id in data["development_claims"]["claim_id"]:
    context = assemble_claim_context(data=data, claim_id=claim_id)

    if context["policy_eligibility"]["policy_incident_status"] in [
        "OUTSIDE_COVERAGE_PERIOD",
        "SUSPENDED_ON_INCIDENT_DATE",
    ]:
        contexts.append(
            {
                "claim_id": claim_id,
                "policy_id": context["policy"]["policy_id"],
                "incident_date": context["claim"]["reported_incident_date"],
                "status": context["policy_eligibility"]["policy_incident_status"],
                "reason": context["policy_eligibility"]["reason"],
            }
        )

pd.DataFrame(contexts).head(10)

,claim_id,policy_id,incident_date,status,reason
0,CLM-000196,POL-00005,2025-07-16,OUTSIDE_COVERAGE_PERIOD,Incident date falls outside the policy coverag...
1,CLM-000198,POL-00018,2026-02-12,OUTSIDE_COVERAGE_PERIOD,Incident date falls outside the policy coverag...
2,CLM-000199,POL-00009,2025-07-03,OUTSIDE_COVERAGE_PERIOD,Incident date falls outside the policy coverag...


In [34]:
outside_coverage_context = assemble_claim_context(
    data=data,
    claim_id="CLM-000198",
)

print("Claim:")
pprint(outside_coverage_context["claim"])

print("\nPolicy:")
pprint(
    {
        key: outside_coverage_context["policy"][key]
        for key in [
            "policy_id",
            "plan_id",
            "coverage_start_date",
            "coverage_end_date",
            "policy_status",
        ]
    }
)

print("\nEligibility assessment:")
pprint(outside_coverage_context["policy_eligibility"])

Claim:
{'claim_category_selected': 'SCREEN_DAMAGE',
 'claim_id': 'CLM-000198',
 'claimed_device_identifier': 'DVC-00000018',
 'customer_id_provided': 'CUS-0033',
 'evidence_bundle_id': 'EBND-000198',
 'policy_id_provided': 'POL-00018',
 'reported_incident_date': '2026-02-12',
 'requested_service_type': 'UNSPECIFIED'}

Policy:
{'coverage_end_date': '2025-12-24',
 'coverage_start_date': '2024-12-25',
 'plan_id': 'DP-ESSENTIAL',
 'policy_id': 'POL-00018',
 'policy_status': 'EXPIRED'}

Eligibility assessment:
{'policy_active_for_incident': False,
 'policy_incident_status': 'OUTSIDE_COVERAGE_PERIOD',
 'reason': 'Incident date falls outside the policy coverage period.'}


In [35]:
context = assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

pprint(context["device_match"])

{'claimed_device_identifier': 'DVC-00000018',
 'covered_device_identifier': 'DVC-00000018',
 'status': 'DEVICE_MATCH'}


In [36]:
print(data["policy_rules"].columns.tolist())

['rule_id', 'rule_name', 'rule_domain', 'priority', 'applies_to_plan_id', 'applies_to_claim_category', 'applies_when', 'required_input_fields', 'pass_condition', 'failure_condition', 'missing_or_unknown_handling', 'outcome_on_failure', 'manual_review_reason', 'policy_reference', 'severity', 'rationale', 'rule_version', 'effective_start_date', 'effective_end_date', 'active_flag']


In [37]:
rule_view_columns = [
    "rule_id",
    "rule_name",
    "priority",
    "rule_type",
    "condition_description",
    "outcome_on_failure",
    "rule_category",
]

available_columns = [
    column
    for column in rule_view_columns
    if column in data["policy_rules"].columns
]

display(
    data["policy_rules"][available_columns]
    .sort_values(["priority", "rule_id"])
)

,rule_id,rule_name,priority,outcome_on_failure
0,DATA-001,Unique policy eligibility record available,10,INFO_REQUIRED
1,DATA-002,Conflicting or multiple policy records,10,MANUAL_REVIEW
2,DATA-003,Plan version and coverage configuration available,10,MANUAL_REVIEW
7,DEV-002,Unresolved claimed-device mismatch,10,MANUAL_REVIEW
5,ELG-003,Supported product and operating scope,10,MANUAL_REVIEW
17,EVD-002,Evidence internally consistent and authenticat...,10,MANUAL_REVIEW
14,EXC-002,Suspected or unclear exclusion,10,MANUAL_REVIEW
10,COV-001,Claim category covered by enrolled plan,20,NOT_ELIGIBLE
8,DEV-003,Verified device not enrolled,20,NOT_ELIGIBLE
4,ELG-002,Policy active on incident date,20,NOT_ELIGIBLE


In [38]:
print("Risk results columns:")
print(data["risk_results"].columns.tolist())

print("\nRisk indicator status distribution:")
for column in data["risk_results"].columns:
    if "status" in column.lower() or "trigger" in column.lower():
        print(f"\n{column}:")
        display(
            data["risk_results"][column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="count")
        )

display(data["risk_results"])

Risk results columns:
['risk_result_id', 'claim_id', 'dataset_partition', 'risk_indicator_id', 'risk_indicator_name', 'risk_status', 'priority', 'recommended_action', 'manual_review_reason', 'source_system', 'source_reference_type', 'source_reference_id', 'related_event_date', 'observed_value', 'threshold_or_window', 'risk_summary', 'calculated_as_of_date', 'calculation_version', 'record_snapshot_date']

Risk indicator status distribution:

risk_status:


,risk_status,count
0,TRIGGERED,4


,risk_result_id,claim_id,dataset_partition,risk_indicator_id,risk_indicator_name,risk_status,priority,recommended_action,manual_review_reason,source_system,source_reference_type,source_reference_id,related_event_date,observed_value,threshold_or_window,risk_summary,calculated_as_of_date,calculation_version,record_snapshot_date
0,RISK-RES-0001,CLM-000183,DEVELOPMENT,RSK-001,HIGH_REPAIR_COST,TRIGGERED,HIGH,MANUAL_REVIEW,HIGH_COST_EXCEPTION,CLAIM_EVIDENCE_METADATA,EVIDENCE_DOCUMENT,EVI-000181,2026-06-23,899.00 USD repair estimate,584.35 USD (65% of 899.00 USD Lumen Lumen Pro ...,Repair estimate exceeds the configured 65% ref...,2026-06-23,v1,2026-06-23
1,RISK-RES-0002,CLM-000184,DEVELOPMENT,RSK-001,HIGH_REPAIR_COST,TRIGGERED,HIGH,MANUAL_REVIEW,HIGH_COST_EXCEPTION,CLAIM_EVIDENCE_METADATA,EVIDENCE_DOCUMENT,EVI-000183,2026-04-05,829.00 USD repair estimate,584.35 USD (65% of 899.00 USD Lumen Lumen Pro ...,Repair estimate exceeds the configured 65% ref...,2026-04-05,v1,2026-06-23
3,RISK-RES-0004,CLM-000180,DEVELOPMENT,RSK-002,RECENT_RELATED_CLAIM_PATTERN,TRIGGERED,MEDIUM,MANUAL_REVIEW,POTENTIAL_DUPLICATE,CASE_PATTERN_RECONCILIATION,HISTORICAL_CLAIM,HCLM-000014,2026-03-18,Related prior claim event 25 calendar days bef...,30 calendar days plus corroborated relation si...,A recent related-claim pattern was identified ...,2026-04-12,v1,2026-06-23
4,RISK-RES-0005,CLM-000181,DEVELOPMENT,RSK-002,RECENT_RELATED_CLAIM_PATTERN,TRIGGERED,MEDIUM,MANUAL_REVIEW,POTENTIAL_DUPLICATE,CASE_PATTERN_RECONCILIATION,PRIOR_SUBMITTED_CLAIM,CLM-000119,2025-07-28,Related prior claim event 22 calendar days bef...,30 calendar days plus corroborated relation si...,A recent related-claim pattern was identified ...,2025-08-19,v1,2026-06-23


In [39]:
risk_context = assemble_claim_context(
    data=data,
    claim_id="CLM-000183",
)

pprint(risk_context["risk"])

{'has_triggered_risk': True,
 'manual_review_reasons': ['HIGH_COST_EXCEPTION'],
 'recommended_action': 'MANUAL_REVIEW',
 'risk_indicator_ids': ['RSK-001'],
 'risk_indicator_names': ['HIGH_REPAIR_COST']}


In [40]:
print(data["policy_rules"].columns.tolist())

['rule_id', 'rule_name', 'rule_domain', 'priority', 'applies_to_plan_id', 'applies_to_claim_category', 'applies_when', 'required_input_fields', 'pass_condition', 'failure_condition', 'missing_or_unknown_handling', 'outcome_on_failure', 'manual_review_reason', 'policy_reference', 'severity', 'rationale', 'rule_version', 'effective_start_date', 'effective_end_date', 'active_flag']


In [41]:
rule_ids_to_review = [
    "ELG-003",
    "EXC-001",
    "EXC-002",
    "DATA-003",
    "OUT-001",
]

display(
    data["policy_rules"]
    .query("rule_id in @rule_ids_to_review")
    .T
)

,2,5,13,14,18
rule_id,DATA-003,ELG-003,EXC-001,EXC-002,OUT-001
rule_name,Plan version and coverage configuration available,Supported product and operating scope,Verified global exclusion does not apply,Suspected or unclear exclusion,Standard processing gate
rule_domain,Data Validation,Eligibility,Exclusion,Exclusion,Outcome
priority,10,10,20,10,60
applies_to_plan_id,ALL,ALL,ALL,ALL,ALL
applies_to_claim_category,ALL,ALL,ALL,ALL,ALL
applies_when,A unique policy record has been found.,A policy record is available.,Available evidence or a verified structured cl...,Narrative or evidence suggests an exclusion bu...,All preceding applicable rules have been evalu...
required_input_fields,plan_id; plan version/effective dates; plan co...,product_family; device_category; operating_reg...,exclusion_status; exclusion_type; verified_evi...,exclusion_status; exclusion_type; narrative_co...,all applicable rule outcomes
pass_condition,An active plan version and corresponding cover...,The policy belongs to the configured DevicePro...,exclusion_status is NOT_APPLICABLE or no exclu...,"No suspected exclusion exists, or the concern ...","No applicable rule returns MANUAL_REVIEW, INFO..."
failure_condition,"The plan, plan version, or coverage configurat...",The policy belongs to an unsupported product f...,"A global exclusion is verified: LOSS, COSMETIC...",A potential exclusion is indicated but evidenc...,One or more preceding applicable rules does no...


In [42]:
from pathlib import Path

reference_dir = PROJECT_ROOT / "data" / "runtime" / "reference"

reference_files = sorted(
    file.name
    for file in reference_dir.iterdir()
    if file.is_file()
)

for file_name in reference_files:
    print(file_name)

active_version_register_v1_1.csv
claim_intake_data_dictionary_v1.csv
decision_precedence_v1_1.csv
decision_taxonomy_dispositions_v1.csv
decision_taxonomy_v1.json
device_value_reference_v1.csv
evidence_metadata_data_dictionary_v1.csv
evidence_profile_requirements_v1.csv
evidence_type_catalog_v1.csv
follow_up_question_catalog_v1.csv
follow_up_question_data_dictionary_v1.csv
follow_up_question_selection_rules_v1.csv
ground_truth_labels_data_dictionary_v1.csv
knowledge_base_manifest_data_dictionary_v1.csv
knowledge_base_manifest_v1.csv
label_source_version_manifest_v1.csv
manual_review_reasons_v1.csv
plan_coverage_matrix_v1.csv
plan_master_data_dictionary_v1.csv
policy_eligibility_lookup_data_dictionary_v1.csv
policy_eligibility_lookup_v1_1.csv
policy_rule_catalog_data_dictionary_v1.csv
policy_rule_catalog_v1_2.csv
prior_claims_history_data_dictionary_v1.csv
prior_claims_history_v1_2.csv
project_data_profile_v1_1.csv
protection_plan_master_v1_1.csv
risk_anomaly_data_dictionary_v1.csv
risk_

In [43]:
matching_files = [
    file_name
    for file_name in reference_files
    if any(
        keyword in file_name.lower()
        for keyword in [
            "exclusion",
            "scope",
            "product",
            "device",
            "plan",
            "dictionary",
        ]
    )
]

print("\nPotentially relevant files:")
for file_name in matching_files:
    print("-", file_name)


Potentially relevant files:
- claim_intake_data_dictionary_v1.csv
- device_value_reference_v1.csv
- evidence_metadata_data_dictionary_v1.csv
- follow_up_question_data_dictionary_v1.csv
- ground_truth_labels_data_dictionary_v1.csv
- knowledge_base_manifest_data_dictionary_v1.csv
- plan_coverage_matrix_v1.csv
- plan_master_data_dictionary_v1.csv
- policy_eligibility_lookup_data_dictionary_v1.csv
- policy_rule_catalog_data_dictionary_v1.csv
- prior_claims_history_data_dictionary_v1.csv
- protection_plan_master_v1_1.csv
- risk_anomaly_data_dictionary_v1.csv
- safety_test_data_dictionary_v1.csv


In [44]:
print(data["plan_master"].columns.tolist())

display(data["plan_master"])

['plan_id', 'plan_version', 'plan_name', 'product_family', 'plan_tier', 'operating_market', 'covered_device_category', 'policy_duration_months', 'annual_claim_limit', 'maximum_theft_claims', 'default_resolution_strategy', 'replacement_allowed_if', 'plan_status', 'effective_start_date', 'effective_end_date', 'plan_description']


,plan_id,plan_version,plan_name,product_family,plan_tier,operating_market,covered_device_category,policy_duration_months,annual_claim_limit,maximum_theft_claims,default_resolution_strategy,replacement_allowed_if,plan_status,effective_start_date,effective_end_date,plan_description
0,DP-ESSENTIAL,1.1,DeviceProtect Essential,DeviceProtect,ESSENTIAL,US,SMARTPHONE,12,1,0,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,Entry-level protection for accidental screen d...
1,DP-COMPLETE,1.1,DeviceProtect Complete,DeviceProtect,COMPLETE,US,SMARTPHONE,12,2,0,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,"Standard protection for accidental, liquid, an..."
2,DP-PREMIUM,1.1,DeviceProtect Premium,DeviceProtect,PREMIUM,US,SMARTPHONE,12,2,1,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,"Comprehensive protection for accidental, liqui..."


In [45]:
display(
    data["policy_lookup"][
        ["device_category", "device_brand", "device_model"]
    ]
    .drop_duplicates()
    .sort_values(["device_category", "device_brand", "device_model"])
)

,device_category,device_brand,device_model
8,SMARTPHONE,Aster,Aster One
0,SMARTPHONE,Cobalt,Cobalt Mini
2,SMARTPHONE,Lumen,Lumen Pro
6,SMARTPHONE,Novatek,Pulse 12
1,SMARTPHONE,Orion,X5
3,SMARTPHONE,Vela,Vela Edge


In [46]:
print(data["plan_master"].columns.tolist())

display(data["plan_master"])

['plan_id', 'plan_version', 'plan_name', 'product_family', 'plan_tier', 'operating_market', 'covered_device_category', 'policy_duration_months', 'annual_claim_limit', 'maximum_theft_claims', 'default_resolution_strategy', 'replacement_allowed_if', 'plan_status', 'effective_start_date', 'effective_end_date', 'plan_description']


,plan_id,plan_version,plan_name,product_family,plan_tier,operating_market,covered_device_category,policy_duration_months,annual_claim_limit,maximum_theft_claims,default_resolution_strategy,replacement_allowed_if,plan_status,effective_start_date,effective_end_date,plan_description
0,DP-ESSENTIAL,1.1,DeviceProtect Essential,DeviceProtect,ESSENTIAL,US,SMARTPHONE,12,1,0,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,Entry-level protection for accidental screen d...
1,DP-COMPLETE,1.1,DeviceProtect Complete,DeviceProtect,COMPLETE,US,SMARTPHONE,12,2,0,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,"Standard protection for accidental, liquid, an..."
2,DP-PREMIUM,1.1,DeviceProtect Premium,DeviceProtect,PREMIUM,US,SMARTPHONE,12,2,1,REPAIR_FIRST,REPAIR_NOT_FEASIBLE,ACTIVE,2024-01-01,2027-12-31,"Comprehensive protection for accidental, liqui..."


In [47]:
# ============================================================
# Plan Configuration Component — Step 1: Data Inspection
# ============================================================

plan_configuration_columns = [
    "plan_id",
    "plan_version",
    "plan_name",
    "product_family",
    "covered_device_category",
    "annual_claim_limit",
    "maximum_theft_claims",
    "plan_status",
    "effective_start_date",
    "effective_end_date",
]

print("Plan configuration fields:")
print(plan_master[plan_configuration_columns].columns.tolist())

print("\nPlan configuration records:")
display(
    plan_master[plan_configuration_columns]
    .sort_values(["plan_id", "plan_version"])
)

print("\nDuplicate plan ID check:")
display(
    plan_master["plan_id"]
    .value_counts()
    .rename_axis("plan_id")
    .reset_index(name="configuration_record_count")
)

print("\nDistinct configuration values:")
for column in [
    "product_family",
    "covered_device_category",
    "plan_status",
]:
    print(f"{column}: {sorted(plan_master[column].dropna().unique().tolist())}")

print("\nEffective date parsing check:")
plan_date_check = plan_master[
    ["plan_id", "effective_start_date", "effective_end_date"]
].copy()

plan_date_check["effective_start_parsed"] = pd.to_datetime(
    plan_date_check["effective_start_date"],
    errors="coerce",
)

plan_date_check["effective_end_parsed"] = pd.to_datetime(
    plan_date_check["effective_end_date"],
    errors="coerce",
)

display(plan_date_check)

Plan configuration fields:
['plan_id', 'plan_version', 'plan_name', 'product_family', 'covered_device_category', 'annual_claim_limit', 'maximum_theft_claims', 'plan_status', 'effective_start_date', 'effective_end_date']

Plan configuration records:


,plan_id,plan_version,plan_name,product_family,covered_device_category,annual_claim_limit,maximum_theft_claims,plan_status,effective_start_date,effective_end_date
1,DP-COMPLETE,1.1,DeviceProtect Complete,DeviceProtect,SMARTPHONE,2,0,ACTIVE,2024-01-01,2027-12-31
0,DP-ESSENTIAL,1.1,DeviceProtect Essential,DeviceProtect,SMARTPHONE,1,0,ACTIVE,2024-01-01,2027-12-31
2,DP-PREMIUM,1.1,DeviceProtect Premium,DeviceProtect,SMARTPHONE,2,1,ACTIVE,2024-01-01,2027-12-31



Duplicate plan ID check:


,plan_id,configuration_record_count
0,DP-ESSENTIAL,1
1,DP-COMPLETE,1
2,DP-PREMIUM,1



Distinct configuration values:
product_family: ['DeviceProtect']
covered_device_category: ['SMARTPHONE']
plan_status: ['ACTIVE']

Effective date parsing check:


,plan_id,effective_start_date,effective_end_date,effective_start_parsed,effective_end_parsed
0,DP-ESSENTIAL,2024-01-01,2027-12-31,2024-01-01,2027-12-31
1,DP-COMPLETE,2024-01-01,2027-12-31,2024-01-01,2027-12-31
2,DP-PREMIUM,2024-01-01,2027-12-31,2024-01-01,2027-12-31


In [48]:
# ============================================================
# Plan Configuration Component — Step 2A: Inspect Claim Columns
# ============================================================

print("Development claims columns:")
print(data["development_claims"].columns.tolist())

print("\nFirst development claim:")
display(data["development_claims"].head(1).T)

Development claims columns:
['claim_id', 'case_created_date', 'claim_submission_channel', 'claim_language', 'policy_id_provided', 'customer_id_provided', 'claimed_device_identifier', 'reported_incident_date', 'claim_category_selected', 'requested_service_type', 'evidence_bundle_id', 'customer_statement', 'intake_record_status', 'record_snapshot_date']

First development claim:


,0
claim_id,CLM-000001
case_created_date,2025-07-14
claim_submission_channel,CONTACT_CENTER
claim_language,EN
policy_id_provided,POL-00018
customer_id_provided,CUS-0033
claimed_device_identifier,DVC-00000018
reported_incident_date,2025-07-14
claim_category_selected,SCREEN_DAMAGE
requested_service_type,REPAIR


In [49]:
# ============================================================
# Plan Configuration Component — Step 2: Happy-Path Check
# ============================================================

from pprint import pprint

from src.tools.policy_lookup import (
    find_policy_records,
    classify_lookup_result,
)
from src.tools.plan_configuration import assess_plan_configuration


sample_claim = data["development_claims"].iloc[0]

policy_matches = find_policy_records(
    policy_lookup=data["policy_lookup"],
    policy_id=sample_claim["policy_id_provided"],
    customer_id=sample_claim["customer_id_provided"],
    device_identifier=sample_claim["claimed_device_identifier"],
)

policy_lookup_status = classify_lookup_result(policy_matches)

print("Claim ID:", sample_claim["claim_id"])
print("Reported incident date:", sample_claim["reported_incident_date"])
print("Policy lookup status:", policy_lookup_status)

if policy_lookup_status != "UNIQUE_MATCH":
    print(
        "\nPlan configuration cannot be assessed because "
        "this claim does not have one uniquely matched policy."
    )

else:
    matched_policy = policy_matches.iloc[0]
    matched_plan_id = matched_policy["plan_id"]

    print("Matched policy ID:", matched_policy["policy_id"])
    print("Matched plan ID:", matched_plan_id)

    plan_configuration_result = assess_plan_configuration(
        plan_master=data["plan_master"],
        plan_id=matched_plan_id,
        incident_date=sample_claim["reported_incident_date"],
    )

    print("\nPlan configuration status:")
    print(plan_configuration_result["plan_configuration_status"])

    print("\nPlan configuration available:")
    print(plan_configuration_result["plan_configuration_available"])

    print("\nProduct scope status:")
    print(plan_configuration_result["product_scope_status"])

    print("\nProduct scope supported:")
    print(plan_configuration_result["product_scope_supported"])

    print("\nReason:")
    print(plan_configuration_result["reason"])

    print("\nSelected plan configuration:")
    pprint(plan_configuration_result["plan_configuration"])

Claim ID: CLM-000001
Reported incident date: 2025-07-14
Policy lookup status: UNIQUE_MATCH
Matched policy ID: POL-00018
Matched plan ID: DP-ESSENTIAL

Plan configuration status:
ACTIVE_CONFIGURATION_AVAILABLE

Plan configuration available:
True

Product scope status:
IN_SCOPE

Product scope supported:
True

Reason:
One active plan configuration is effective on the incident date.

Selected plan configuration:
{'annual_claim_limit': 1,
 'covered_device_category': 'SMARTPHONE',
 'effective_end_date': '2027-12-31',
 'effective_start_date': '2024-01-01',
 'maximum_theft_claims': 0,
 'plan_id': 'DP-ESSENTIAL',
 'plan_name': 'DeviceProtect Essential',
 'plan_status': 'ACTIVE',
 'plan_version': 1.1,
 'product_family': 'DeviceProtect'}


In [50]:
# ============================================================
# Plan Configuration Component — Step 3: Controlled Exceptions
# ============================================================

exception_checks = {
    "Missing plan ID": {
        "plan_id": None,
        "incident_date": "2025-07-14",
    },
    "Incident date before plan effective period": {
        "plan_id": "DP-ESSENTIAL",
        "incident_date": "2023-12-31",
    },
    "Invalid incident date": {
        "plan_id": "DP-ESSENTIAL",
        "incident_date": "not-a-valid-date",
    },
}

for test_name, test_inputs in exception_checks.items():
    result = assess_plan_configuration(
        plan_master=data["plan_master"],
        plan_id=test_inputs["plan_id"],
        incident_date=test_inputs["incident_date"],
    )

    print(f"\n{'=' * 70}")
    print(test_name)
    print(f"{'=' * 70}")
    print("Plan configuration status:", result["plan_configuration_status"])
    print("Plan configuration available:", result["plan_configuration_available"])
    print("Product scope status:", result["product_scope_status"])
    print("Reason:", result["reason"])


Missing plan ID
Plan configuration status: NOT_ASSESSED
Plan configuration available: None
Product scope status: NOT_ASSESSED
Reason: A uniquely matched policy plan ID is required.

Incident date before plan effective period
Plan configuration status: NO_EFFECTIVE_CONFIGURATION
Plan configuration available: None
Product scope status: NOT_ASSESSED
Reason: No plan configuration is effective on the reported incident date.

Invalid incident date
Plan configuration status: INCIDENT_DATE_MISSING_OR_INVALID
Plan configuration available: None
Product scope status: NOT_ASSESSED
Reason: Incident date is required to identify the applicable plan configuration.


In [51]:
# ============================================================
# Plan Configuration Component — Step 4: Overlapping Configurations
# ============================================================



overlapping_plan_master = data["plan_master"].copy()

duplicate_configuration = overlapping_plan_master[
    overlapping_plan_master["plan_id"] == "DP-ESSENTIAL"
].copy()

duplicate_configuration["plan_version"] = 2.0
duplicate_configuration["plan_name"] = "DeviceProtect Essential - Test Version"
duplicate_configuration["effective_start_date"] = "2025-01-01"
duplicate_configuration["effective_end_date"] = "2026-12-31"

overlapping_plan_master = pd.concat(
    [overlapping_plan_master, duplicate_configuration],
    ignore_index=True,
)

overlap_result = assess_plan_configuration(
    plan_master=overlapping_plan_master,
    plan_id="DP-ESSENTIAL",
    incident_date="2025-07-14",
)

print("Plan configuration status:", overlap_result["plan_configuration_status"])
print("Plan configuration available:", overlap_result["plan_configuration_available"])
print("Product scope status:", overlap_result["product_scope_status"])
print("Reason:", overlap_result["reason"])

print("\nOverlapping records created for this controlled test:")
display(
    overlapping_plan_master[
        overlapping_plan_master["plan_id"] == "DP-ESSENTIAL"
    ][
        [
            "plan_id",
            "plan_version",
            "plan_name",
            "effective_start_date",
            "effective_end_date",
            "plan_status",
        ]
    ]
)

Plan configuration status: MULTIPLE_EFFECTIVE_CONFIGURATIONS
Plan configuration available: None
Product scope status: NOT_ASSESSED
Reason: More than one plan configuration is effective on the reported incident date.

Overlapping records created for this controlled test:


,plan_id,plan_version,plan_name,effective_start_date,effective_end_date,plan_status
0,DP-ESSENTIAL,1.1,DeviceProtect Essential,2024-01-01,2027-12-31,ACTIVE
3,DP-ESSENTIAL,2.0,DeviceProtect Essential - Test Version,2025-01-01,2026-12-31,ACTIVE


In [52]:
# ============================================================
# Plan Configuration Component — Step 5: Scope and Status Tests
# ============================================================

scope_test_plan_id = "DP-ESSENTIAL"
scope_test_date = "2025-07-14"

test_cases = {
    "Product family outside project scope": {
        "column_to_change": "product_family",
        "new_value": "OtherProtect",
    },
    "Device category outside project scope": {
        "column_to_change": "covered_device_category",
        "new_value": "TABLET",
    },
    "Inactive configuration": {
        "column_to_change": "plan_status",
        "new_value": "INACTIVE",
    },
}

for test_name, change in test_cases.items():
    test_plan_master = data["plan_master"].copy()

    target_row = test_plan_master["plan_id"] == scope_test_plan_id

    test_plan_master.loc[
        target_row,
        change["column_to_change"],
    ] = change["new_value"]

    result = assess_plan_configuration(
        plan_master=test_plan_master,
        plan_id=scope_test_plan_id,
        incident_date=scope_test_date,
    )

    print(f"\n{'=' * 70}")
    print(test_name)
    print(f"{'=' * 70}")
    print("Plan configuration status:", result["plan_configuration_status"])
    print("Plan configuration available:", result["plan_configuration_available"])
    print("Product scope status:", result["product_scope_status"])
    print("Product scope supported:", result["product_scope_supported"])
    print("Reason:", result["reason"])

    print("\nSelected configuration:")
    pprint(result["plan_configuration"])


Product family outside project scope
Plan configuration status: ACTIVE_CONFIGURATION_AVAILABLE
Plan configuration available: True
Product scope status: PRODUCT_FAMILY_OUT_OF_SCOPE
Product scope supported: False
Reason: One active plan configuration is effective on the incident date.

Selected configuration:
{'annual_claim_limit': 1,
 'covered_device_category': 'SMARTPHONE',
 'effective_end_date': '2027-12-31',
 'effective_start_date': '2024-01-01',
 'maximum_theft_claims': 0,
 'plan_id': 'DP-ESSENTIAL',
 'plan_name': 'DeviceProtect Essential',
 'plan_status': 'ACTIVE',
 'plan_version': 1.1,
 'product_family': 'OtherProtect'}

Device category outside project scope
Plan configuration status: ACTIVE_CONFIGURATION_AVAILABLE
Plan configuration available: True
Product scope status: DEVICE_CATEGORY_OUT_OF_SCOPE
Product scope supported: False
Reason: One active plan configuration is effective on the incident date.

Selected configuration:
{'annual_claim_limit': 1,
 'covered_device_category': 

In [53]:
# ============================================================
# Plan Configuration Component — Step 6: Context Integration Check
# ============================================================

import importlib
from pprint import pprint

import src.claim_context as claim_context_module

# Reload is important because claim_context.py was edited
importlib.reload(claim_context_module)

assemble_claim_context = claim_context_module.assemble_claim_context

context = assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

print("Claim ID:", context["claim"]["claim_id"])
print("Policy lookup status:", context["policy_lookup_status"])

print("\nPlan configuration status:")
print(context["plan_configuration"]["plan_configuration_status"])

print("\nPlan configuration available:")
print(context["plan_configuration"]["plan_configuration_available"])

print("\nProduct scope status:")
print(context["plan_configuration"]["product_scope_status"])

print("\nProduct scope supported:")
print(context["plan_configuration"]["product_scope_supported"])

print("\nSelected plan configuration:")
pprint(context["plan_configuration"]["plan_configuration"])

Claim ID: CLM-000001
Policy lookup status: UNIQUE_MATCH

Plan configuration status:
ACTIVE_CONFIGURATION_AVAILABLE

Plan configuration available:
True

Product scope status:
IN_SCOPE

Product scope supported:
True

Selected plan configuration:
{'annual_claim_limit': 1,
 'covered_device_category': 'SMARTPHONE',
 'effective_end_date': '2027-12-31',
 'effective_start_date': '2024-01-01',
 'maximum_theft_claims': 0,
 'plan_id': 'DP-ESSENTIAL',
 'plan_name': 'DeviceProtect Essential',
 'plan_status': 'ACTIVE',
 'plan_version': 1.1,
 'product_family': 'DeviceProtect'}


In [54]:
import importlib
from pprint import pprint

import src.claim_context as claim_context_module

importlib.reload(claim_context_module)

context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

print("Claim ID:", context["claim"]["claim_id"])
print("Policy lookup:", context["policy_lookup_status"])

print("\nPlan configuration:")
pprint(context["plan_configuration"])

Claim ID: CLM-000001
Policy lookup: UNIQUE_MATCH

Plan configuration:
{'plan_configuration': {'annual_claim_limit': 1,
                        'covered_device_category': 'SMARTPHONE',
                        'effective_end_date': '2027-12-31',
                        'effective_start_date': '2024-01-01',
                        'maximum_theft_claims': 0,
                        'plan_id': 'DP-ESSENTIAL',
                        'plan_name': 'DeviceProtect Essential',
                        'plan_status': 'ACTIVE',
                        'plan_version': 1.1,
                        'product_family': 'DeviceProtect'},
 'plan_configuration_available': True,
 'plan_configuration_status': 'ACTIVE_CONFIGURATION_AVAILABLE',
 'product_scope_status': 'IN_SCOPE',
 'product_scope_supported': True,
 'reason': 'One active plan configuration is effective on the incident date.'}


In [55]:
# ============================================================
# Claim Context — Full Summary Check
# ============================================================

from pprint import pprint

context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

context_summary = {
    "claim_id": context["claim"]["claim_id"],
    "reported_incident_date": context["claim"]["reported_incident_date"],

    "policy_lookup_status": context["policy_lookup_status"],

    "policy_incident_status": context["policy_eligibility"][
        "policy_incident_status"
    ],
    "policy_active_for_incident": context["policy_eligibility"][
        "policy_active_for_incident"
    ],

    "plan_configuration_status": context["plan_configuration"][
        "plan_configuration_status"
    ],
    "product_scope_status": context["plan_configuration"][
        "product_scope_status"
    ],

    "device_match_status": context["device_match"]["status"],

    "coverage_lookup_status": context["coverage"][
        "coverage_lookup_status"
    ],
    "covered_flag": context["coverage"]["covered_flag"],

    "annual_claims_used": context["claim_history"][
        "annual_claims_used"
    ],
    "theft_claims_used": context["claim_history"][
        "theft_claims_used"
    ],

    "evidence_assessment_status": context["evidence"][
        "evidence_assessment_status"
    ],

    "has_triggered_risk": context["risk"]["has_triggered_risk"],
    "risk_recommended_action": context["risk"]["recommended_action"],
}

pprint(context_summary)

{'annual_claims_used': 0,
 'claim_id': 'CLM-000001',
 'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD',
 'covered_flag': True,
 'device_match_status': 'DEVICE_MATCH',
 'evidence_assessment_status': 'SUFFICIENT',
 'has_triggered_risk': False,
 'plan_configuration_status': 'ACTIVE_CONFIGURATION_AVAILABLE',
 'policy_active_for_incident': True,
 'policy_incident_status': 'ACTIVE_ON_INCIDENT_DATE',
 'policy_lookup_status': 'UNIQUE_MATCH',
 'product_scope_status': 'IN_SCOPE',
 'reported_incident_date': '2025-07-14',
 'risk_recommended_action': None,
 'theft_claims_used': 0}


In [56]:
# ============================================================
# Plan Configuration Component — Step 7: Development Batch Check
# ============================================================

from src.tools.policy_lookup import (
    classify_lookup_result,
    find_policy_records,
)
from src.tools.plan_configuration import assess_plan_configuration

plan_configuration_batch_results = []

for _, claim in data["development_claims"].iterrows():

    policy_records = find_policy_records(
        policy_lookup=data["policy_lookup"],
        policy_id=claim["policy_id_provided"],
        customer_id=claim["customer_id_provided"],
        device_identifier=claim["claimed_device_identifier"],
    )

    policy_lookup_status = classify_lookup_result(policy_records)

    plan_id = None

    if policy_lookup_status == "UNIQUE_MATCH":
        plan_id = policy_records.iloc[0]["plan_id"]

    result = assess_plan_configuration(
        plan_master=data["plan_master"],
        plan_id=plan_id,
        incident_date=claim["reported_incident_date"],
    )

    plan_configuration_batch_results.append(
        {
            "claim_id": claim["claim_id"],
            "policy_lookup_status": policy_lookup_status,
            "plan_id": plan_id,
            "plan_configuration_status": result[
                "plan_configuration_status"
            ],
            "plan_configuration_available": result[
                "plan_configuration_available"
            ],
            "product_scope_status": result[
                "product_scope_status"
            ],
            "product_scope_supported": result[
                "product_scope_supported"
            ],
        }
    )

plan_configuration_batch_df = pd.DataFrame(
    plan_configuration_batch_results
)

print("Plan configuration status distribution:")
display(
    plan_configuration_batch_df[
        "plan_configuration_status"
    ].value_counts(dropna=False)
)

print("\nProduct scope status distribution:")
display(
    plan_configuration_batch_df[
        "product_scope_status"
    ].value_counts(dropna=False)
)

print("\nClaims with configuration issues or unavailable assessment:")
display(
    plan_configuration_batch_df[
        plan_configuration_batch_df[
            "plan_configuration_status"
        ] != "ACTIVE_CONFIGURATION_AVAILABLE"
    ].sort_values("claim_id")
)

Plan configuration status distribution:


plan_configuration_status
ACTIVE_CONFIGURATION_AVAILABLE      148
INCIDENT_DATE_MISSING_OR_INVALID     11
NOT_ASSESSED                          6
Name: count, dtype: int64


Product scope status distribution:


product_scope_status
IN_SCOPE        148
NOT_ASSESSED     17
Name: count, dtype: int64


Claims with configuration issues or unavailable assessment:


,claim_id,policy_lookup_status,plan_id,plan_configuration_status,plan_configuration_available,product_scope_status,product_scope_supported
53,CLM-000071,UNIQUE_MATCH,DP-PREMIUM,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
54,CLM-000072,UNIQUE_MATCH,DP-COMPLETE,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
55,CLM-000073,UNIQUE_MATCH,DP-PREMIUM,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
56,CLM-000074,UNIQUE_MATCH,DP-ESSENTIAL,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
57,CLM-000075,UNIQUE_MATCH,DP-PREMIUM,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
58,CLM-000076,UNIQUE_MATCH,DP-COMPLETE,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
59,CLM-000077,UNIQUE_MATCH,DP-ESSENTIAL,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
60,CLM-000078,UNIQUE_MATCH,DP-PREMIUM,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
61,CLM-000079,UNIQUE_MATCH,DP-COMPLETE,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None
62,CLM-000080,UNIQUE_MATCH,DP-ESSENTIAL,INCIDENT_DATE_MISSING_OR_INVALID,None,NOT_ASSESSED,None


In [57]:
# ============================================================
# Plan Configuration Component — Step 8: Invalid Date Audit
# ============================================================

invalid_date_claim_ids = plan_configuration_batch_df.loc[
    plan_configuration_batch_df["plan_configuration_status"]
    == "INCIDENT_DATE_MISSING_OR_INVALID",
    "claim_id",
].tolist()

invalid_date_audit = data["development_claims"].loc[
    data["development_claims"]["claim_id"].isin(invalid_date_claim_ids),
    [
        "claim_id",
        "case_created_date",
        "reported_incident_date",
        "record_snapshot_date",
        "policy_id_provided",
        "claim_category_selected",
    ],
].copy()

invalid_date_audit["reported_incident_date_raw"] = (
    invalid_date_audit["reported_incident_date"]
    .map(repr)
)

invalid_date_audit["reported_incident_date_parsed"] = pd.to_datetime(
    invalid_date_audit["reported_incident_date"],
    errors="coerce",
)

display(
    invalid_date_audit[
        [
            "claim_id",
            "reported_incident_date_raw",
            "reported_incident_date_parsed",
            "case_created_date",
            "policy_id_provided",
            "claim_category_selected",
        ]
    ].sort_values("claim_id")
)

,claim_id,reported_incident_date_raw,reported_incident_date_parsed,case_created_date,policy_id_provided,claim_category_selected
53,CLM-000071,nan,NaT,2026-06-17,POL-00045,MECHANICAL_BREAKDOWN
54,CLM-000072,nan,NaT,2026-01-16,POL-00100,MECHANICAL_BREAKDOWN
55,CLM-000073,nan,NaT,2026-05-12,POL-00021,SCREEN_DAMAGE
56,CLM-000074,nan,NaT,2026-01-11,POL-00037,SCREEN_DAMAGE
57,CLM-000075,nan,NaT,2025-09-15,POL-00105,LIQUID_DAMAGE
58,CLM-000076,nan,NaT,2026-05-24,POL-00034,MECHANICAL_BREAKDOWN
59,CLM-000077,nan,NaT,2026-04-06,POL-00085,SCREEN_DAMAGE
60,CLM-000078,nan,NaT,2026-02-11,POL-00090,SCREEN_DAMAGE
61,CLM-000079,nan,NaT,2026-03-28,POL-00056,ACCIDENTAL_DAMAGE
62,CLM-000080,nan,NaT,2026-06-22,POL-00037,SCREEN_DAMAGE


In [58]:
# ============================================================
# Plan Configuration Component — Step 9: Incomplete Limit Check
# ============================================================

incomplete_plan_master = data["plan_master"].copy()

incomplete_plan_master.loc[
    incomplete_plan_master["plan_id"] == "DP-ESSENTIAL",
    "annual_claim_limit",
] = None

incomplete_result = assess_plan_configuration(
    plan_master=incomplete_plan_master,
    plan_id="DP-ESSENTIAL",
    incident_date="2025-07-14",
)

print(
    "Plan configuration status:",
    incomplete_result["plan_configuration_status"],
)
print(
    "Plan configuration available:",
    incomplete_result["plan_configuration_available"],
)
print(
    "Product scope status:",
    incomplete_result["product_scope_status"],
)
print("Reason:", incomplete_result["reason"])

print("\nReturned configuration:")
pprint(incomplete_result["plan_configuration"])

Plan configuration status: CONFIGURATION_INCOMPLETE
Plan configuration available: False
Product scope status: NOT_ASSESSED
Reason: Annual claim limit or theft claim limit is missing, invalid, or negative.

Returned configuration:
{'annual_claim_limit': None,
 'covered_device_category': 'SMARTPHONE',
 'effective_end_date': '2027-12-31',
 'effective_start_date': '2024-01-01',
 'maximum_theft_claims': 0,
 'plan_id': 'DP-ESSENTIAL',
 'plan_name': 'DeviceProtect Essential',
 'plan_status': 'ACTIVE',
 'plan_version': 1.1,
 'product_family': 'DeviceProtect'}


In [59]:
# ============================================================
# Deterministic Triage Engine — Step 1: Policy Rule Catalogue
# ============================================================

policy_rules = data["policy_rules"].copy()

print("Policy rule catalogue columns:")
print(policy_rules.columns.tolist())

print("\nPolicy rule catalogue shape:")
print(policy_rules.shape)

print("\nAll policy rules:")
display(policy_rules)

core_rule_ids = [
    "DATA-001",
    "DATA-002",
    "DATA-003",
    "DEV-001",
    "DEV-002",
    "DEV-003",
    "ELG-001",
    "ELG-002",
    "ELG-003",
    "COV-001",
    "LIM-001",
    "LIM-002",
    "CLM-001",
    "EVD-001",
    "EVD-002",
    "ANM-001",
    "OUT-001",
]

print("\nCore deterministic rules:")
display(
    policy_rules[
        policy_rules["rule_id"].isin(core_rule_ids)
    ].sort_values("rule_id")
)

Policy rule catalogue columns:
['rule_id', 'rule_name', 'rule_domain', 'priority', 'applies_to_plan_id', 'applies_to_claim_category', 'applies_when', 'required_input_fields', 'pass_condition', 'failure_condition', 'missing_or_unknown_handling', 'outcome_on_failure', 'manual_review_reason', 'policy_reference', 'severity', 'rationale', 'rule_version', 'effective_start_date', 'effective_end_date', 'active_flag']

Policy rule catalogue shape:
(19, 20)

All policy rules:


,rule_id,rule_name,rule_domain,priority,applies_to_plan_id,applies_to_claim_category,applies_when,required_input_fields,pass_condition,failure_condition,missing_or_unknown_handling,outcome_on_failure,manual_review_reason,policy_reference,severity,rationale,rule_version,effective_start_date,effective_end_date,active_flag
0,DATA-001,Unique policy eligibility record available,Data Validation,10,ALL,ALL,A claim is submitted for assessment.,customer_id OR policy_id; policy eligibility l...,Exactly one policy eligibility record is match...,No policy eligibility record can be matched us...,Request the missing or corrected customer/poli...,INFO_REQUIRED,NaN,DATA-LOOKUP-01,BLOCKING,The system cannot assess coverage without a un...,1.0,2026-06-22,NaN,Yes
1,DATA-002,Conflicting or multiple policy records,Data Validation,10,ALL,ALL,More than one candidate policy record is retur...,policy eligibility lookup result; source-syste...,One authoritative policy record is returned an...,"Multiple matching records, duplicate active po...",Do not ask the customer to resolve a system co...,MANUAL_REVIEW,DATA_CONFLICT,DATA-LOOKUP-02,BLOCKING,The agent must not choose among conflicting sy...,1.0,2026-06-22,NaN,Yes
2,DATA-003,Plan version and coverage configuration available,Data Validation,10,ALL,ALL,A unique policy record has been found.,plan_id; plan version/effective dates; plan co...,An active plan version and corresponding cover...,"The plan, plan version, or coverage configurat...",Do not infer missing product terms; refer to a...,MANUAL_REVIEW,RULE_NOT_AVAILABLE,DATA-PLAN-01,BLOCKING,"Coverage cannot be assessed without a valid, e...",1.0,2026-06-22,NaN,Yes
3,ELG-001,Incident date present and valid,Eligibility,30,ALL,ALL,The claim category is potentially assessable.,incident_date,"Incident date is present, parseable, and not a...","Incident date is missing, ambiguous, malformed...",Request the date on which the incident occurre...,INFO_REQUIRED,NaN,ELG-DATE-01,BLOCKING,Coverage-period and waiting-period checks requ...,1.0,2026-06-22,NaN,Yes
4,ELG-002,Policy active on incident date,Eligibility,20,ALL,ALL,A valid incident date and policy record are av...,coverage_start_date; coverage_end_date; policy...,Incident date falls within the policy coverage...,"Incident date is before coverage start, after ...",If status history is incomplete or contradicto...,NOT_ELIGIBLE,NaN,ELG-ACTIVE-01,BLOCKING,An inactive policy is a conclusive coverage fa...,1.0,2026-06-22,NaN,Yes
5,ELG-003,Supported product and operating scope,Eligibility,10,ALL,ALL,A policy record is available.,product_family; device_category; operating_reg...,The policy belongs to the configured DevicePro...,The policy belongs to an unsupported product f...,Unsupported or unclear scope is referred to an...,MANUAL_REVIEW,RULE_NOT_AVAILABLE,ELG-SCOPE-01,BLOCKING,The MVP must not apply smartphone-product rule...,1.0,2026-06-22,NaN,Yes
6,DEV-001,Claimed device identifier available,Device Validation,30,ALL,ALL,A claim refers to a covered smartphone.,claimed_device_identifier (IMEI or serial number),The claim provides a usable IMEI or serial num...,"Device identifier is missing, malformed, or no...",Request the IMEI or serial number shown on the...,INFO_REQUIRED,NaN,DEV-ID-01,BLOCKING,Device-specific coverage requires a reliable d...,1.0,2026-06-22,NaN,Yes
7,DEV-002,Unresolved claimed-device mismatch,Device Validation,10,ALL,ALL,A usable claimed device identifier and enrolle...,claimed_device_identifier; enrolled_device_ide...,Claimed device matches the enrolled device or ...,"Claimed device does not match, but the mismatc...",Refer for analyst validation rather than rejec...,MANUAL_REVIEW,DEVICE_MISMATCH,DEV-MATCH-01,BLOCKING,Some mismatches are correctable operational-da...,1.0,2026-06-22,NaN,Yes
8,DEV-003,Verified device not enrolled,Device Validation,20,ALL,ALL,An analyst-quality device match decision is av...,claimed_device_identifier; enrolled_device_ide...,Verified device match status is MATCH


Core deterministic rules:


,rule_id,rule_name,rule_domain,priority,applies_to_plan_id,applies_to_claim_category,applies_when,required_input_fields,pass_condition,failure_condition,missing_or_unknown_handling,outcome_on_failure,manual_review_reason,policy_reference,severity,rationale,rule_version,effective_start_date,effective_end_date,active_flag
15,ANM-001,Risk or anomaly review trigger,Anomaly,40,ALL,ALL,Eligibility and hard coverage checks have passed.,risk_indicator_list; duplicate_claim_flag; rec...,No active material anomaly or review trigger i...,"A material anomaly is present, such as potenti...",Risk signals never create a decline. Absence o...,MANUAL_REVIEW,NaN,ANM-REVIEW-01,REVIEW,Anomalies are routing signals only and must no...,1.2,2026-06-22,NaN,Yes
9,CLM-001,Claim category identifiable,Claim Classification,30,ALL,ALL,A claim narrative or structured intake has bee...,claim_category OR claim_description,One supported claim category can be identified...,"Category is absent, ambiguous, or cannot be ma...",Ask a targeted question about what happened an...,INFO_REQUIRED,NaN,CLM-CAT-01,BLOCKING,Coverage and evidence requirements depend on t...,1.0,2026-06-22,NaN,Yes
10,COV-001,Claim category covered by enrolled plan,Coverage,20,ALL,ALL,A valid plan and claim category are available.,plan_id; claim_category; plan_coverage_matrix....,The coverage matrix shows covered_flag = Yes f...,The coverage matrix shows covered_flag = No fo...,"If the matrix is unavailable, apply DATA-003; ...",NOT_ELIGIBLE,NaN,COV-MATRIX-01,BLOCKING,Explicit product coverage is a conclusive poli...,1.0,2026-06-22,NaN,Yes
0,DATA-001,Unique policy eligibility record available,Data Validation,10,ALL,ALL,A claim is submitted for assessment.,customer_id OR policy_id; policy eligibility l...,Exactly one policy eligibility record is match...,No policy eligibility record can be matched us...,Request the missing or corrected customer/poli...,INFO_REQUIRED,NaN,DATA-LOOKUP-01,BLOCKING,The system cannot assess coverage without a un...,1.0,2026-06-22,NaN,Yes
1,DATA-002,Conflicting or multiple policy records,Data Validation,10,ALL,ALL,More than one candidate policy record is retur...,policy eligibility lookup result; source-syste...,One authoritative policy record is returned an...,"Multiple matching records, duplicate active po...",Do not ask the customer to resolve a system co...,MANUAL_REVIEW,DATA_CONFLICT,DATA-LOOKUP-02,BLOCKING,The agent must not choose among conflicting sy...,1.0,2026-06-22,NaN,Yes
2,DATA-003,Plan version and coverage configuration available,Data Validation,10,ALL,ALL,A unique policy record has been found.,plan_id; plan version/effective dates; plan co...,An active plan version and corresponding cover...,"The plan, plan version, or coverage configurat...",Do not infer missing product terms; refer to a...,MANUAL_REVIEW,RULE_NOT_AVAILABLE,DATA-PLAN-01,BLOCKING,"Coverage cannot be assessed without a valid, e...",1.0,2026-06-22,NaN,Yes
6,DEV-001,Claimed device identifier available,Device Validation,30,ALL,ALL,A claim refers to a covered smartphone.,claimed_device_identifier (IMEI or serial number),The claim provides a usable IMEI or serial num...,"Device identifier is missing, malformed, or no...",Request the IMEI or serial number shown on the...,INFO_REQUIRED,NaN,DEV-ID-01,BLOCKING,Device-specific coverage requires a reliable d...,1.0,2026-06-22,NaN,Yes
7,DEV-002,Unresolved claimed-device mismatch,Device Validation,10,ALL,ALL,A usable claimed device identifier and enrolle...,claimed_device_identifier; enrolled_device_ide...,Claimed device matches the enrolled device or ...,"Claimed device does not match, but the mismatc...",Refer for analyst validation rather than rejec...,MANUAL_REVIEW,DEVICE_MISMATCH,DEV-MATCH-01,BLOCKING,Some mismatches are correctable operational-da...,1.0,2026-06-22,NaN,Yes
8,DEV-003,Verified device not enrolled,Device Validation,20,ALL,ALL,An analyst-quality device match decision is av...,claimed_device_identifier; enrolled_device_ide...,Verified 

In [60]:
# ============================================================
# Deterministic Triage Engine — Step 2: Context Status Audit
# ============================================================

from pprint import pprint

coverage_matrix = data["coverage_matrix"].copy()

print("Coverage flag datatype:")
print(coverage_matrix["covered_flag"].dtype)

print("\nCoverage flag values and Python types:")
coverage_flag_audit = coverage_matrix[
    ["plan_id", "claim_category", "covered_flag", "evidence_profile_id"]
].copy()

coverage_flag_audit["python_type"] = coverage_flag_audit[
    "covered_flag"
].map(lambda value: type(value).__name__)

display(coverage_flag_audit.sort_values(["plan_id", "claim_category"]))

print("\nDistinct covered_flag values:")
print(
    coverage_matrix["covered_flag"]
    .value_counts(dropna=False)
)

# Build one status row per development claim using the full assembler.
context_status_rows = []

for claim_id in data["development_claims"]["claim_id"]:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    context_status_rows.append(
        {
            "claim_id": claim_id,
            "policy_lookup_status": context["policy_lookup_status"],
            "policy_incident_status": context["policy_eligibility"][
                "policy_incident_status"
            ],
            "plan_configuration_status": context["plan_configuration"][
                "plan_configuration_status"
            ],
            "product_scope_status": context["plan_configuration"][
                "product_scope_status"
            ],
            "device_match_status": context["device_match"]["status"],
            "coverage_lookup_status": context["coverage"][
                "coverage_lookup_status"
            ],
            "covered_flag": context["coverage"]["covered_flag"],
            "evidence_assessment_status": context["evidence"][
                "evidence_assessment_status"
            ],
            "has_triggered_risk": context["risk"]["has_triggered_risk"],
        }
    )

context_status_audit = pd.DataFrame(context_status_rows)

status_columns = [
    "policy_lookup_status",
    "policy_incident_status",
    "plan_configuration_status",
    "product_scope_status",
    "device_match_status",
    "coverage_lookup_status",
    "covered_flag",
    "evidence_assessment_status",
    "has_triggered_risk",
]

for column in status_columns:
    print(f"\n{'=' * 70}")
    print(f"{column} distribution")
    print(f"{'=' * 70}")
    display(
        context_status_audit[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="claim_count")
    )

Coverage flag datatype:
bool

Coverage flag values and Python types:


,plan_id,claim_category,covered_flag,evidence_profile_id,python_type
7,DP-COMPLETE,ACCIDENTAL_DAMAGE,True,EVD-ACCIDENTAL-01,bool
8,DP-COMPLETE,LIQUID_DAMAGE,True,EVD-LIQUID-01,bool
11,DP-COMPLETE,LOSS,False,NaN,bool
9,DP-COMPLETE,MECHANICAL_BREAKDOWN,True,EVD-MECHANICAL-01,bool
6,DP-COMPLETE,SCREEN_DAMAGE,True,EVD-SCREEN-01,bool
10,DP-COMPLETE,THEFT,False,NaN,bool
1,DP-ESSENTIAL,ACCIDENTAL_DAMAGE,False,NaN,bool
2,DP-ESSENTIAL,LIQUID_DAMAGE,False,NaN,bool
5,DP-ESSENTIAL,LOSS,False,NaN,bool
3,DP-ESSENTIAL,MECHANICAL_BREAKDOWN,False,NaN,bool



Distinct covered_flag values:
covered_flag
True     10
False     8
Name: count, dtype: int64

policy_lookup_status distribution


,policy_lookup_status,claim_count
0,UNIQUE_MATCH,159
1,MULTIPLE_MATCH,6



policy_incident_status distribution


...

,covered_flag,claim_count
0,True,135
1,None,15
2,False,15



evidence_assessment_status distribution


,evidence_assessment_status,claim_count
0,SUFFICIENT,113
1,NOT_APPLICABLE,30
2,INCOMPLETE,11
3,CONTRADICTORY,11



has_triggered_risk distribution


,has_triggered_risk,claim_count
0,False,161
1,True,4


In [61]:
# ============================================================
# Deterministic Triage Engine — Step 3: Dependency Audit
# ============================================================

claim_fields = data["development_claims"][
    [
        "claim_id",
        "reported_incident_date",
        "claim_category_selected",
        "policy_id_provided",
        "claimed_device_identifier",
        "evidence_bundle_id",
    ]
].copy()

dependency_audit = (
    context_status_audit
    .merge(claim_fields, on="claim_id", how="left")
    .sort_values("claim_id")
)

audit_columns = [
    "claim_id",
    "policy_lookup_status",
    "policy_incident_status",
    "plan_configuration_status",
    "device_match_status",
    "claim_category_selected",
    "coverage_lookup_status",
    "covered_flag",
    "evidence_assessment_status",
    "has_triggered_risk",
]

print("1. Claims with no coverage record")
display(
    dependency_audit.loc[
        dependency_audit["coverage_lookup_status"]
        == "NO_COVERAGE_RECORD",
        audit_columns,
    ]
)

print("\n2. Claims with explicitly uncovered categories")
display(
    dependency_audit.loc[
        dependency_audit["covered_flag"] == False,
        audit_columns,
    ]
)

print("\n3. Claims where evidence is not applicable")
display(
    dependency_audit.loc[
        dependency_audit["evidence_assessment_status"]
        == "NOT_APPLICABLE",
        audit_columns,
    ]
)

print("\n4. Claims outside the policy coverage period")
display(
    dependency_audit.loc[
        dependency_audit["policy_incident_status"]
        == "OUTSIDE_COVERAGE_PERIOD",
        audit_columns,
    ]
)

print("\n5. Claims with a triggered risk signal")
display(
    dependency_audit.loc[
        dependency_audit["has_triggered_risk"] == True,
        audit_columns,
    ]
)

1. Claims with no coverage record


,claim_id,policy_lookup_status,policy_incident_status,plan_configuration_status,device_match_status,claim_category_selected,coverage_lookup_status,covered_flag,evidence_assessment_status,has_triggered_risk
64,CLM-000086,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
65,CLM-000087,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
66,CLM-000088,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
67,CLM-000089,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
68,CLM-000090,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
69,CLM-000091,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
70,CLM-000092,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
71,CLM-000093,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
72,CLM-000094,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
122,CLM-000162,MULTIPLE_MATCH,NOT_ASSESSED,NOT_ASSESSED,NOT_ASSESSED,SCREEN_DAMAGE,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False



2. Claims with explicitly uncovered categories


,claim_id,policy_lookup_status,policy_incident_status,plan_configuration_status,device_match_status,claim_category_selected,coverage_lookup_status,covered_flag,evidence_assessment_status,has_triggered_risk
116,CLM-000154,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,THEFT,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
118,CLM-000156,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,THEFT,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
119,CLM-000157,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,THEFT,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
139,CLM-000186,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,LOSS,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
140,CLM-000187,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,LOSS,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
141,CLM-000188,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,LOSS,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
142,CLM-000189,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,LOSS,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
143,CLM-000190,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,THEFT,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
144,CLM-000191,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,THEFT,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False
145,CLM-000192,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,LOSS,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False



3. Claims where evidence is not applicable


,claim_id,policy_lookup_status,policy_incident_status,plan_configuration_status,device_match_status,claim_category_selected,coverage_lookup_status,covered_flag,evidence_assessment_status,has_triggered_risk
64,CLM-000086,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
65,CLM-000087,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
66,CLM-000088,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
67,CLM-000089,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
68,CLM-000090,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
69,CLM-000091,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
70,CLM-000092,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
71,CLM-000093,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
72,CLM-000094,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,UNSPECIFIED,NO_COVERAGE_RECORD,None,NOT_APPLICABLE,False
116,CLM-000154,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,THEFT,UNIQUE_COVERAGE_RECORD,False,NOT_APPLICABLE,False



4. Claims outside the policy coverage period


,claim_id,policy_lookup_status,policy_incident_status,plan_configuration_status,device_match_status,claim_category_selected,coverage_lookup_status,covered_flag,evidence_assessment_status,has_triggered_risk
147,CLM-000196,UNIQUE_MATCH,OUTSIDE_COVERAGE_PERIOD,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,SUFFICIENT,False
149,CLM-000198,UNIQUE_MATCH,OUTSIDE_COVERAGE_PERIOD,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,SUFFICIENT,False
150,CLM-000199,UNIQUE_MATCH,OUTSIDE_COVERAGE_PERIOD,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,SUFFICIENT,False



5. Claims with a triggered risk signal


,claim_id,policy_lookup_status,policy_incident_status,plan_configuration_status,device_match_status,claim_category_selected,coverage_lookup_status,covered_flag,evidence_assessment_status,has_triggered_risk
135,CLM-000180,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,MECHANICAL_BREAKDOWN,UNIQUE_COVERAGE_RECORD,True,SUFFICIENT,True
136,CLM-000181,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,SUFFICIENT,True
137,CLM-000183,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,MECHANICAL_BREAKDOWN,UNIQUE_COVERAGE_RECORD,True,SUFFICIENT,True
138,CLM-000184,UNIQUE_MATCH,ACTIVE_ON_INCIDENT_DATE,ACTIVE_CONFIGURATION_AVAILABLE,DEVICE_MATCH,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,SUFFICIENT,True


In [62]:
# ============================================================
# Deterministic Triage Engine — Step 4: Claim-Limit History Audit
# ============================================================

from src.tools.claims_history import (
    calculate_claim_usage,
    get_policy_claim_history,
)

history_audit_rows = []

for claim_id in data["development_claims"]["claim_id"]:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    policy_record = context["policy"]
    incident_date = context["claim"]["reported_incident_date"]

    # Only audit cases where policy and incident date are usable.
    if (
        policy_record is None
        or context["policy_eligibility"]["policy_incident_status"]
        not in {"ACTIVE_ON_INCIDENT_DATE", "OUTSIDE_COVERAGE_PERIOD"}
    ):
        continue

    incident_date_parsed = pd.to_datetime(
        incident_date,
        errors="coerce",
    )

    coverage_start = pd.to_datetime(
        policy_record["coverage_start_date"],
        errors="coerce",
    )

    coverage_end = pd.to_datetime(
        policy_record["coverage_end_date"],
        errors="coerce",
    )

    # Current implementation: prior claims before incident date.
    current_history = get_policy_claim_history(
        prior_claims=data["prior_claims"],
        policy_id=policy_record["policy_id"],
        incident_date=incident_date,
    )

    current_usage = calculate_claim_usage(current_history)

    # Policy-term logic: prior claims before incident date and
    # within the matched policy's coverage period.
    policy_term_history = data["prior_claims"].copy()

    policy_term_history["incident_date_parsed"] = pd.to_datetime(
        policy_term_history["incident_date"],
        errors="coerce",
    )

    policy_term_history = policy_term_history[
        (policy_term_history["policy_id"] == policy_record["policy_id"])
        & (
            policy_term_history["incident_date_parsed"]
            >= coverage_start
        )
        & (
            policy_term_history["incident_date_parsed"]
            <= coverage_end
        )
        & (
            policy_term_history["incident_date_parsed"]
            < incident_date_parsed
        )
    ].copy()

    policy_term_usage = calculate_claim_usage(policy_term_history)

    history_audit_rows.append(
        {
            "claim_id": claim_id,
            "policy_id": policy_record["policy_id"],
            "incident_date": incident_date,
            "coverage_start_date": policy_record["coverage_start_date"],
            "coverage_end_date": policy_record["coverage_end_date"],
            "current_history_count": len(current_history),
            "policy_term_history_count": len(policy_term_history),
            "current_annual_usage": current_usage["annual_claims_used"],
            "policy_term_annual_usage": policy_term_usage[
                "annual_claims_used"
            ],
            "difference_in_annual_usage": (
                current_usage["annual_claims_used"]
                - policy_term_usage["annual_claims_used"]
            ),
            "current_theft_usage": current_usage["theft_claims_used"],
            "policy_term_theft_usage": policy_term_usage[
                "theft_claims_used"
            ],
            "difference_in_theft_usage": (
                current_usage["theft_claims_used"]
                - policy_term_usage["theft_claims_used"]
            ),
        }
    )

history_audit_df = pd.DataFrame(history_audit_rows)

difference_rows = history_audit_df[
    (history_audit_df["difference_in_annual_usage"] != 0)
    | (history_audit_df["difference_in_theft_usage"] != 0)
].sort_values("claim_id")

print("Claims audited:", len(history_audit_df))
print("Claims with usage differences:", len(difference_rows))

display(difference_rows)

Claims audited: 148
Claims with usage differences: 0


,claim_id,policy_id,incident_date,coverage_start_date,coverage_end_date,current_history_count,policy_term_history_count,current_annual_usage,policy_term_annual_usage,difference_in_annual_usage,current_theft_usage,policy_term_theft_usage,difference_in_theft_usage


In [63]:
# ============================================================
# Deterministic Triage Engine — Step 5: Baseline Rule Mapping
# ============================================================

triage_rule_mapping = pd.DataFrame(
    [
        {
            "precedence_stage": 1,
            "rule_id": "DATA-001",
            "context_field": "policy_lookup_status",
            "trigger_condition": "NO_MATCH",
            "triage_outcome": "INFO_REQUIRED",
            "reason": "No uniquely usable policy can be identified.",
        },
        {
            "precedence_stage": 1,
            "rule_id": "DATA-002",
            "context_field": "policy_lookup_status",
            "trigger_condition": "MULTIPLE_MATCH",
            "triage_outcome": "MANUAL_REVIEW",
            "reason": "System must not choose between multiple policies.",
        },
        {
            "precedence_stage": 1,
            "rule_id": "DATA-003",
            "context_field": "plan_configuration_status / coverage_lookup_status",
            "trigger_condition": (
                "Configuration unavailable, incomplete, conflicting, "
                "or coverage record unexpectedly unavailable"
            ),
            "triage_outcome": "MANUAL_REVIEW",
            "reason": "Do not infer plan or coverage terms.",
        },
        {
            "precedence_stage": 2,
            "rule_id": "ELG-002",
            "context_field": "policy_eligibility.policy_incident_status",
            "trigger_condition": "OUTSIDE_COVERAGE_PERIOD",
            "triage_outcome": "NOT_ELIGIBLE",
            "reason": "Policy was not active on the incident date.",
        },
        {
            "precedence_stage": 2,
            "rule_id": "COV-001",
            "context_field": "coverage.covered_flag",
            "trigger_condition": "False with UNIQUE_COVERAGE_RECORD",
            "triage_outcome": "NOT_ELIGIBLE",
            "reason": "Claim category is explicitly not covered.",
        },
        {
            "precedence_stage": 2,
            "rule_id": "LIM-001",
            "context_field": "claim_history / plan_configuration",
            "trigger_condition": "annual_claims_used >= annual_claim_limit",
            "triage_outcome": "NOT_ELIGIBLE",
            "reason": "Annual claim allowance is exhausted.",
        },
        {
            "precedence_stage": 2,
            "rule_id": "LIM-002",
            "context_field": "claim_history / plan_configuration",
            "trigger_condition": (
                "THEFT and theft_claims_used >= maximum_theft_claims"
            ),
            "triage_outcome": "NOT_ELIGIBLE",
            "reason": "Theft claim allowance is exhausted.",
        },
        {
            "precedence_stage": 3,
            "rule_id": "ELG-001",
            "context_field": "policy_eligibility.policy_incident_status",
            "trigger_condition": "INCIDENT_DATE_MISSING_OR_INVALID",
            "triage_outcome": "INFO_REQUIRED",
            "reason": "Incident date is needed for eligibility assessment.",
        },
        {
            "precedence_stage": 3,
            "rule_id": "DEV-001",
            "context_field": "device_match.status",
            "trigger_condition": "DEVICE_IDENTIFIER_MISSING",
            "triage_outcome": "INFO_REQUIRED",
            "reason": "A device identifier is required.",
        },
        {
            "precedence_stage": 3,
            "rule_id": "CLM-001",
            "context_field": "claim.claim_category_selected",
            "trigger_condition": "Missing, blank, or UNSPECIFIED",
            "triage_outcome": "INFO_REQUIRED",
            "reason": "Claim category is required for coverage assessment.",
        },
        {
            "precedence_stage": 4,
            "rule_id": "ELG-003",
            "context_field": "plan_configuration.product_scope_status",
            "trigger_condition": "Not IN_SCOPE",
            "triage_outcome": "MANUAL_REVIEW",
            "reason": "Unsupported or unclear product scope.",
        },
        {
            "precedence_stage": 4,
            "rule_id": "DEV-002",
            "context_field": "device_match.status",
            "trigger_condition": "DEVICE_MISMATCH",
            "triage_outcome": "MANUAL_REVIEW",
            "reason": "Claimed device mismatch needs analyst validation.",
        },
        {
            "precedence_stage": 4,
            "rule_id": "EVD-002",
            "context_field": "evidence.evidence_assessment_status",
            "trigger_condition": "CONTRADICTORY",
            "triage_outcome": "MANUAL_REVIEW",
            "reason": "Contradictory evidence must not be resolved automatically.",
        },
        {
            "precedence_stage": 5,
            "rule_id": "ANM-001",
            "context_field": "risk.has_triggered_risk",
            "trigger_condition": "True",
            "triage_outcome": "MANUAL_REVIEW",
            "reason": "Risk indicators are routing signals only.",
        },
        {
            "precedence_stage": 6,
            "rule_id": "EVD-001",
            "context_field": "evidence.evidence_assessment_status",
            "trigger_condition": "INCOMPLETE",
            "triage_outcome": "INFO_REQUIRED",
            "reason": "Required evidence is missing or unreadable.",
        },
        {
            "precedence_stage": 7,
            "rule_id": "OUT-001",
            "context_field": "all prior checks",
            "trigger_condition": "No earlier rule triggered",
            "triage_outcome": "PROCEED",
            "reason": "Eligible for standard processing only.",
        },
    ]
)

display(
    triage_rule_mapping.sort_values(
        ["precedence_stage", "rule_id"]
    ).reset_index(drop=True)
)

deferred_rules = pd.DataFrame(
    [
        {
            "rule_id": "DEV-003",
            "reason_deferred": (
                "Current device tool identifies mismatch but does not "
                "provide a verified NOT_ENROLLED decision."
            ),
        },
        {
            "rule_id": "EXC-001",
            "reason_deferred": (
                "No structured exclusion-status dataset is available."
            ),
        },
        {
            "rule_id": "EXC-002",
            "reason_deferred": (
                "Narrative-led exclusion assessment is reserved for a "
                "later LLM/human-review workflow."
            ),
        },
    ]
)

print("Deferred rules for the first deterministic baseline:")
display(deferred_rules)

,precedence_stage,rule_id,context_field,trigger_condition,triage_outcome,reason
0,1,DATA-001,policy_lookup_status,NO_MATCH,INFO_REQUIRED,No uniquely usable policy can be identified.
1,1,DATA-002,policy_lookup_status,MULTIPLE_MATCH,MANUAL_REVIEW,System must not choose between multiple policies.
2,1,DATA-003,plan_configuration_status / coverage_lookup_st...,"Configuration unavailable, incomplete, conflic...",MANUAL_REVIEW,Do not infer plan or coverage terms.
3,2,COV-001,coverage.covered_flag,False with UNIQUE_COVERAGE_RECORD,NOT_ELIGIBLE,Claim category is explicitly not covered.
4,2,ELG-002,policy_eligibility.policy_incident_status,OUTSIDE_COVERAGE_PERIOD,NOT_ELIGIBLE,Policy was not active on the incident date.
5,2,LIM-001,claim_history / plan_configuration,annual_claims_used >= annual_claim_limit,NOT_ELIGIBLE,Annual claim allowance is exhausted.
6,2,LIM-002,claim_history / plan_configuration,THEFT and theft_claims_used >= maximum_theft_c...,NOT_ELIGIBLE,Theft claim allowance is exhausted.
7,3,CLM-001,claim.claim_category_selected,"Missing, blank, or UNSPECIFIED",INFO_REQUIRED,Claim category is required for coverage assess...
8,3,DEV-001,device_match.status,DEVICE_IDENTIFIER_MISSING,INFO_REQUIRED,A device identifier is required.
9,3,ELG-001,policy_eligibility.policy_incident_status,INCIDENT_DATE_MISSING_OR_INVALID,INFO_REQUIRED,Incident date is needed for eligibility assess...


Deferred rules for the first deterministic baseline:


,rule_id,reason_deferred
0,DEV-003,Current device tool identifies mismatch but do...
1,EXC-001,No structured exclusion-status dataset is avai...
2,EXC-002,Narrative-led exclusion assessment is reserved...


In [64]:
# ============================================================
# Deterministic Triage Engine — Step 6: Final Baseline Rule Map
# ============================================================

triage_rule_mapping = pd.DataFrame(
    [
        # ----------------------------------------------------
        # Stage 1: Data, configuration, ambiguity, conflict
        # ----------------------------------------------------
        {
            "precedence_stage": 1,
            "rule_id": "DATA-001",
            "trigger_condition": (
                "policy_lookup_status == 'NO_MATCH'"
            ),
            "triage_outcome": "INFO_REQUIRED",
            "reason": (
                "No policy can be identified from the supplied "
                "customer/policy/device details."
            ),
        },
        {
            "precedence_stage": 1,
            "rule_id": "DATA-002",
            "trigger_condition": (
                "policy_lookup_status == 'MULTIPLE_MATCH'"
            ),
            "triage_outcome": "MANUAL_REVIEW",
            "reason": (
                "The system must not choose between multiple "
                "candidate policy records."
            ),
        },
        {
            "precedence_stage": 1,
            "rule_id": "DATA-003",
            "trigger_condition": (
                "Unique policy and valid incident date exist, but "
                "plan configuration is unavailable, incomplete, "
                "inactive, conflicting, or no expected coverage "
                "record exists for an identified category."
            ),
            "triage_outcome": "MANUAL_REVIEW",
            "reason": (
                "The system must not infer plan terms or coverage "
                "when configuration is unavailable."
            ),
        },
        {
            "precedence_stage": 1,
            "rule_id": "ELG-003",
            "trigger_condition": (
                "Active plan configuration exists and "
                "product_scope_status is not 'IN_SCOPE'."
            ),
            "triage_outcome": "MANUAL_REVIEW",
            "reason": (
                "The MVP must not apply DeviceProtect smartphone "
                "rules outside its supported scope."
            ),
        },
        {
            "precedence_stage": 1,
            "rule_id": "DEV-002",
            "trigger_condition": (
                "device_match.status == 'DEVICE_MISMATCH'"
            ),
            "triage_outcome": "MANUAL_REVIEW",
            "reason": (
                "A mismatch can reflect an operational-data issue; "
                "it is not an automatic denial."
            ),
        },
        {
            "precedence_stage": 1,
            "rule_id": "EVD-002",
            "trigger_condition": (
                "evidence.evidence_assessment_status == "
                "'CONTRADICTORY'"
            ),
            "triage_outcome": "MANUAL_REVIEW",
            "reason": (
                "Contradictory evidence must be resolved by a "
                "human reviewer."
            ),
        },

        # ----------------------------------------------------
        # Stage 2: Verified hard eligibility failures
        # ----------------------------------------------------
        {
            "precedence_stage": 2,
            "rule_id": "ELG-002",
            "trigger_condition": (
                "policy_incident_status == "
                "'OUTSIDE_COVERAGE_PERIOD'"
            ),
            "triage_outcome": "NOT_ELIGIBLE",
            "reason": (
                "The policy was not active on the reported "
                "incident date."
            ),
        },
        {
            "precedence_stage": 2,
            "rule_id": "COV-001",
            "trigger_condition": (
                "coverage_lookup_status == "
                "'UNIQUE_COVERAGE_RECORD' and "
                "covered_flag is False"
            ),
            "triage_outcome": "NOT_ELIGIBLE",
            "reason": (
                "The identified claim category is explicitly not "
                "covered by the enrolled plan."
            ),
        },
        {
            "precedence_stage": 2,
            "rule_id": "LIM-001",
            "trigger_condition": (
                "annual_claims_used >= annual_claim_limit"
            ),
            "triage_outcome": "NOT_ELIGIBLE",
            "reason": (
                "The plan's annual claim allowance is exhausted."
            ),
        },
        {
            "precedence_stage": 2,
            "rule_id": "LIM-002",
            "trigger_condition": (
                "Claim category is THEFT and "
                "theft_claims_used >= maximum_theft_claims."
            ),
            "triage_outcome": "NOT_ELIGIBLE",
            "reason": (
                "The plan's theft claim allowance is exhausted."
            ),
        },

        # ----------------------------------------------------
        # Stage 3: Missing core facts
        # ----------------------------------------------------
        {
            "precedence_stage": 3,
            "rule_id": "ELG-001",
            "trigger_condition": (
                "policy_incident_status == "
                "'INCIDENT_DATE_MISSING_OR_INVALID'"
            ),
            "triage_outcome": "INFO_REQUIRED",
            "reason": (
                "A valid incident date is required for "
                "incident-date eligibility assessment."
            ),
        },
        {
            "precedence_stage": 3,
            "rule_id": "DEV-001",
            "trigger_condition": (
                "device_match.status == "
                "'DEVICE_IDENTIFIER_MISSING'"
            ),
            "triage_outcome": "INFO_REQUIRED",
            "reason": (
                "A usable IMEI or serial number is required."
            ),
        },
        {
            "precedence_stage": 3,
            "rule_id": "CLM-001",
            "trigger_condition": (
                "claim category is missing, blank, or "
                "'UNSPECIFIED'."
            ),
            "triage_outcome": "INFO_REQUIRED",
            "reason": (
                "The claim category is required to determine "
                "coverage and evidence requirements."
            ),
        },

        # ----------------------------------------------------
        # Stage 4: Risk and anomaly routing
        # ----------------------------------------------------
        {
            "precedence_stage": 4,
            "rule_id": "ANM-001",
            "trigger_condition": (
                "risk.has_triggered_risk is True"
            ),
            "triage_outcome": "MANUAL_REVIEW",
            "reason": (
                "Risk indicators route the claim for review only; "
                "they never establish fraud or customer fault."
            ),
        },

        # ----------------------------------------------------
        # Stage 5: Required evidence gaps
        # ----------------------------------------------------
        {
            "precedence_stage": 5,
            "rule_id": "EVD-001",
            "trigger_condition": (
                "evidence.evidence_assessment_status == "
                "'INCOMPLETE'"
            ),
            "triage_outcome": "INFO_REQUIRED",
            "reason": (
                "Required evidence is missing or unreadable."
            ),
        },

        # ----------------------------------------------------
        # Stage 6: Final gate
        # ----------------------------------------------------
        {
            "precedence_stage": 6,
            "rule_id": "OUT-001",
            "trigger_condition": (
                "No preceding applicable rule has triggered."
            ),
            "triage_outcome": "PROCEED",
            "reason": (
                "Eligible for standard processing only; this is "
                "not an approval or payout decision."
            ),
        },
    ]
)

display(
    triage_rule_mapping.sort_values(
        ["precedence_stage", "rule_id"]
    ).reset_index(drop=True)
)

deferred_rules = pd.DataFrame(
    [
        {
            "rule_id": "DEV-003",
            "reason_deferred": (
                "Current runtime data does not provide a verified "
                "NOT_ENROLLED device decision."
            ),
        },
        {
            "rule_id": "EXC-001",
            "reason_deferred": (
                "No structured exclusion-status field is available "
                "in the runtime package."
            ),
        },
        {
            "rule_id": "EXC-002",
            "reason_deferred": (
                "Narrative-led exclusion assessment belongs to a "
                "later LLM extraction and human-review workflow."
            ),
        },
    ]
)

print("Deferred rules for the first deterministic baseline:")
display(deferred_rules)

,precedence_stage,rule_id,trigger_condition,triage_outcome,reason
0,1,DATA-001,policy_lookup_status == 'NO_MATCH',INFO_REQUIRED,No policy can be identified from the supplied ...
1,1,DATA-002,policy_lookup_status == 'MULTIPLE_MATCH',MANUAL_REVIEW,The system must not choose between multiple ca...
2,1,DATA-003,"Unique policy and valid incident date exist, b...",MANUAL_REVIEW,The system must not infer plan terms or covera...
3,1,DEV-002,device_match.status == 'DEVICE_MISMATCH',MANUAL_REVIEW,A mismatch can reflect an operational-data iss...
4,1,ELG-003,Active plan configuration exists and product_s...,MANUAL_REVIEW,The MVP must not apply DeviceProtect smartphon...
5,1,EVD-002,evidence.evidence_assessment_status == 'CONTRA...,MANUAL_REVIEW,Contradictory evidence must be resolved by a h...
6,2,COV-001,coverage_lookup_status == 'UNIQUE_COVERAGE_REC...,NOT_ELIGIBLE,The identified claim category is explicitly no...
7,2,ELG-002,policy_incident_status == 'OUTSIDE_COVERAGE_PE...,NOT_ELIGIBLE,The policy was not active on the reported inci...
8,2,LIM-001,annual_claims_used >= annual_claim_limit,NOT_ELIGIBLE,The plan's annual claim allowance is exhausted.
9,2,LIM-002,Claim category is THEFT and theft_claims_used ...,NOT_ELIGIBLE,The plan's theft claim allowance is exhausted.


Deferred rules for the first deterministic baseline:


,rule_id,reason_deferred
0,DEV-003,Current runtime data does not provide a verifi...
1,EXC-001,No structured exclusion-status field is availa...
2,EXC-002,Narrative-led exclusion assessment belongs to ...


In [65]:
# ============================================================
# Deterministic Triage Engine — Step 7: Output Contract
# ============================================================

from pprint import pprint

triage_output_contract = pd.DataFrame(
    [
        {
            "field": "claim_id",
            "type": "str",
            "purpose": "Claim identifier from the assembled context.",
        },
        {
            "field": "triage_outcome",
            "type": "str",
            "purpose": (
                "One of PROCEED, INFO_REQUIRED, MANUAL_REVIEW, "
                "or NOT_ELIGIBLE."
            ),
        },
        {
            "field": "triggering_rule_id",
            "type": "str",
            "purpose": (
                "The first applicable rule under the defined "
                "precedence order. OUT-001 is used for PROCEED."
            ),
        },
        {
            "field": "precedence_stage",
            "type": "int",
            "purpose": (
                "The safety-precedence stage that produced the "
                "triage outcome."
            ),
        },
        {
            "field": "decision_reason",
            "type": "str",
            "purpose": (
                "Clear explanation of why the claim received "
                "this recommendation."
            ),
        },
        {
            "field": "rule_trace",
            "type": "list[dict]",
            "purpose": (
                "All rules evaluated up to and including the "
                "triggering rule."
            ),
        },
        {
            "field": "decision_support_only",
            "type": "bool",
            "purpose": (
                "Always True. The output is not a payment, "
                "settlement, fraud, or final-denial decision."
            ),
        },
        {
            "field": "system_limitations",
            "type": "list[str]",
            "purpose": (
                "Known baseline rules or data capabilities that "
                "are intentionally not evaluated."
            ),
        },
    ]
)

display(triage_output_contract)

rule_execution_order = [
    (1, "DATA-001"),
    (1, "DATA-002"),
    (1, "DATA-003"),
    (1, "ELG-003"),
    (1, "DEV-002"),
    (1, "EVD-002"),
    (2, "ELG-002"),
    (2, "COV-001"),
    (2, "LIM-001"),
    (2, "LIM-002"),
    (3, "ELG-001"),
    (3, "CLM-001"),
    (3, "DEV-001"),
    (4, "ANM-001"),
    (5, "EVD-001"),
    (6, "OUT-001"),
]

execution_order_df = pd.DataFrame(
    rule_execution_order,
    columns=["precedence_stage", "rule_id"],
)

print("Explicit rule execution order:")
display(execution_order_df)

example_proceed_output = {
    "claim_id": "CLM-000001",
    "triage_outcome": "PROCEED",
    "triggering_rule_id": "OUT-001",
    "precedence_stage": 6,
    "decision_reason": (
        "All applicable deterministic checks passed. "
        "Eligible for standard processing only; this is not "
        "an approval or payout decision."
    ),
    "rule_trace": [
        {
            "rule_id": "DATA-001",
            "precedence_stage": 1,
            "evaluation": "NOT_TRIGGERED",
            "observed_value": "UNIQUE_MATCH",
        },
        {
            "rule_id": "ELG-002",
            "precedence_stage": 2,
            "evaluation": "NOT_TRIGGERED",
            "observed_value": "ACTIVE_ON_INCIDENT_DATE",
        },
        {
            "rule_id": "EVD-001",
            "precedence_stage": 5,
            "evaluation": "NOT_TRIGGERED",
            "observed_value": "SUFFICIENT",
        },
        {
            "rule_id": "OUT-001",
            "precedence_stage": 6,
            "evaluation": "TRIGGERED",
            "observed_value": "No earlier applicable rule triggered",
        },
    ],
    "decision_support_only": True,
    "system_limitations": [
        (
            "DEV-003 is not evaluated because the runtime context "
            "does not provide a verified NOT_ENROLLED decision."
        ),
        (
            "EXC-001 and EXC-002 are not evaluated because no "
            "structured exclusion-status dataset is available."
        ),
        (
            "Claim-history source completeness cannot be independently "
            "verified from the current runtime package."
        ),
    ],
}

print("\nExample future output for a clean claim context:")
pprint(example_proceed_output)

,field,type,purpose
0,claim_id,str,Claim identifier from the assembled context.
1,triage_outcome,str,"One of PROCEED, INFO_REQUIRED, MANUAL_REVIEW, ..."
2,triggering_rule_id,str,The first applicable rule under the defined pr...
3,precedence_stage,int,The safety-precedence stage that produced the ...
4,decision_reason,str,Clear explanation of why the claim received th...
5,rule_trace,list[dict],All rules evaluated up to and including the tr...
6,decision_support_only,bool,"Always True. The output is not a payment, sett..."
7,system_limitations,list[str],Known baseline rules or data capabilities that...


Explicit rule execution order:


,precedence_stage,rule_id
0,1,DATA-001
1,1,DATA-002
2,1,DATA-003
3,1,ELG-003
4,1,DEV-002
5,1,EVD-002
6,2,ELG-002
7,2,COV-001
8,2,LIM-001
9,2,LIM-002



Example future output for a clean claim context:
{'claim_id': 'CLM-000001',
 'decision_reason': 'All applicable deterministic checks passed. Eligible for '
                    'standard processing only; this is not an approval or '
                    'payout decision.',
 'decision_support_only': True,
 'precedence_stage': 6,
 'rule_trace': [{'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'UNIQUE_MATCH',
                 'precedence_stage': 1,
                 'rule_id': 'DATA-001'},
                {'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'ACTIVE_ON_INCIDENT_DATE',
                 'precedence_stage': 2,
                 'rule_id': 'ELG-002'},
                {'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'SUFFICIENT',
                 'precedence_stage': 5,
                 'rule_id': 'EVD-001'},
                {'evaluation': 'TRIGGERED',
                 'observed_value': 'No earlier applicable rule triggered',
     

In [66]:
# ============================================================
# Deterministic Triage Engine — Step 9: Stage 1 Validation
# ============================================================

import importlib
from copy import deepcopy

import src.triage_engine as triage_engine_module

importlib.reload(triage_engine_module)

evaluate_stage_one = triage_engine_module.evaluate_stage_one

clean_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

multiple_match_claim_id = context_status_audit.loc[
    context_status_audit["policy_lookup_status"] == "MULTIPLE_MATCH",
    "claim_id",
].iloc[0]

device_mismatch_claim_id = context_status_audit.loc[
    context_status_audit["device_match_status"] == "DEVICE_MISMATCH",
    "claim_id",
].iloc[0]

contradictory_evidence_claim_id = context_status_audit.loc[
    context_status_audit["evidence_assessment_status"] == "CONTRADICTORY",
    "claim_id",
].iloc[0]

multiple_match_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id=multiple_match_claim_id,
)

device_mismatch_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id=device_mismatch_claim_id,
)

contradictory_evidence_context = (
    claim_context_module.assemble_claim_context(
        data=data,
        claim_id=contradictory_evidence_claim_id,
    )
)

no_match_context = deepcopy(clean_context)
no_match_context["policy_lookup_status"] = "NO_MATCH"

incomplete_configuration_context = deepcopy(clean_context)
incomplete_configuration_context["plan_configuration"][
    "plan_configuration_status"
] = "CONFIGURATION_INCOMPLETE"

unsupported_scope_context = deepcopy(clean_context)
unsupported_scope_context["plan_configuration"][
    "product_scope_status"
] = "PRODUCT_FAMILY_OUT_OF_SCOPE"

coverage_dates_unavailable_context = deepcopy(clean_context)
coverage_dates_unavailable_context["policy_eligibility"][
    "policy_incident_status"
] = "COVERAGE_DATES_UNAVAILABLE"

stage_one_scenarios = {
    "Clean claim — no Stage 1 issue": clean_context,
    "Actual multiple policy match": multiple_match_context,
    "Actual device mismatch": device_mismatch_context,
    "Actual contradictory evidence": contradictory_evidence_context,
    "Controlled no policy match": no_match_context,
    "Controlled incomplete plan configuration": (
        incomplete_configuration_context
    ),
    "Controlled unsupported scope": unsupported_scope_context,
    "Controlled unavailable policy coverage dates": (
        coverage_dates_unavailable_context
    ),
}

stage_one_results = []

for scenario_name, scenario_context in stage_one_scenarios.items():
    result = evaluate_stage_one(scenario_context)

    if result is None:
        stage_one_results.append(
            {
                "scenario": scenario_name,
                "triage_outcome": None,
                "triggering_rule_id": None,
                "precedence_stage": None,
                "decision_reason": (
                    "No Stage 1 rule triggered; continue to Stage 2."
                ),
            }
        )
    else:
        stage_one_results.append(
            {
                "scenario": scenario_name,
                "triage_outcome": result["triage_outcome"],
                "triggering_rule_id": result["triggering_rule_id"],
                "precedence_stage": result["precedence_stage"],
                "decision_reason": result["decision_reason"],
            }
        )

stage_one_results_df = pd.DataFrame(stage_one_results)

display(stage_one_results_df)

print("Example Stage 1 rule trace — device mismatch:")
device_mismatch_result = evaluate_stage_one(device_mismatch_context)

for trace_item in device_mismatch_result["rule_trace"]:
    print(trace_item)

,scenario,triage_outcome,triggering_rule_id,precedence_stage,decision_reason
0,Clean claim — no Stage 1 issue,NaN,NaN,NaN,No Stage 1 rule triggered; continue to Stage 2.
1,Actual multiple policy match,MANUAL_REVIEW,DATA-002,1.0,Multiple candidate policy records were found. ...
2,Actual device mismatch,MANUAL_REVIEW,DEV-002,1.0,The claimed device does not match the enrolled...
3,Actual contradictory evidence,MANUAL_REVIEW,EVD-002,1.0,Submitted evidence contains a contradiction an...
4,Controlled no policy match,INFO_REQUIRED,DATA-001,1.0,No policy can be identified from the supplied ...
5,Controlled incomplete plan configuration,MANUAL_REVIEW,DATA-003,1.0,"Plan or coverage configuration is unavailable,..."
6,Controlled unsupported scope,MANUAL_REVIEW,ELG-003,1.0,"The selected plan configuration is outside, or..."
7,Controlled unavailable policy coverage dates,MANUAL_REVIEW,ELG-002,1.0,"Policy coverage dates are unavailable, so inci..."


Example Stage 1 rule trace — device mismatch:
{'rule_id': 'DATA-001', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'DATA-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'ELG-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'ACTIVE_ON_INCIDENT_DATE'}
{'rule_id': 'DATA-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': {'plan_configuration_status': 'ACTIVE_CONFIGURATION_AVAILABLE', 'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD'}}
{'rule_id': 'ELG-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'IN_SCOPE'}
{'rule_id': 'DEV-002', 'precedence_stage': 1, 'evaluation': 'TRIGGERED', 'observed_value': 'DEVICE_MISMATCH'}


In [67]:
# ============================================================
# Deterministic Triage Engine — Trace Retention Check
# ============================================================

import importlib
from pprint import pprint

import src.triage_engine as triage_engine_module

importlib.reload(triage_engine_module)

evaluate_stage_one = triage_engine_module.evaluate_stage_one

clean_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

shared_rule_trace = []

stage_one_decision = evaluate_stage_one(
    clean_context,
    rule_trace=shared_rule_trace,
)

print("Stage 1 decision:", stage_one_decision)
print("Trace records retained:", len(shared_rule_trace))

pprint(shared_rule_trace)

Stage 1 decision: None
Trace records retained: 7
[{'evaluation': 'NOT_TRIGGERED',
  'observed_value': 'UNIQUE_MATCH',
  'precedence_stage': 1,
  'rule_id': 'DATA-001'},
 {'evaluation': 'NOT_TRIGGERED',
  'observed_value': 'UNIQUE_MATCH',
  'precedence_stage': 1,
  'rule_id': 'DATA-002'},
 {'evaluation': 'NOT_TRIGGERED',
  'observed_value': 'ACTIVE_ON_INCIDENT_DATE',
  'precedence_stage': 1,
  'rule_id': 'ELG-002'},
 {'evaluation': 'NOT_TRIGGERED',
  'observed_value': {'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD',
                     'plan_configuration_status': 'ACTIVE_CONFIGURATION_AVAILABLE'},
  'precedence_stage': 1,
  'rule_id': 'DATA-003'},
 {'evaluation': 'NOT_TRIGGERED',
  'observed_value': 'IN_SCOPE',
  'precedence_stage': 1,
  'rule_id': 'ELG-003'},
 {'evaluation': 'NOT_TRIGGERED',
  'observed_value': 'DEVICE_MATCH',
  'precedence_stage': 1,
  'rule_id': 'DEV-002'},
 {'evaluation': 'NOT_TRIGGERED',
  'observed_value': 'SUFFICIENT',
  'precedence_stage': 1,
  'rule_id': 

In [68]:
# ============================================================
# Deterministic Triage Engine — Step 10: Stage 2 Limit Audit
# ============================================================

stage_two_audit_rows = []

for claim_id in data["development_claims"]["claim_id"]:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    stage_one_trace = []
    stage_one_decision = evaluate_stage_one(
        context,
        rule_trace=stage_one_trace,
    )

    claim = context["claim"]
    policy_eligibility = context["policy_eligibility"]
    plan_configuration = context["plan_configuration"]
    selected_plan = plan_configuration.get("plan_configuration") or {}
    coverage = context["coverage"]
    claim_history = context["claim_history"]

    annual_claim_limit = selected_plan.get("annual_claim_limit")
    maximum_theft_claims = selected_plan.get("maximum_theft_claims")

    annual_claims_used = claim_history["annual_claims_used"]
    theft_claims_used = claim_history["theft_claims_used"]

    claim_category = claim["claim_category_selected"]
    plan_id = selected_plan.get("plan_id")

    annual_limit_exhausted = (
        annual_claim_limit is not None
        and annual_claims_used >= annual_claim_limit
    )

    theft_limit_applicable = (
        plan_id == "DP-PREMIUM"
        and claim_category == "THEFT"
    )

    theft_limit_exhausted = (
        theft_limit_applicable
        and maximum_theft_claims is not None
        and theft_claims_used >= maximum_theft_claims
    )

    stage_two_audit_rows.append(
        {
            "claim_id": claim_id,
            "stage_one_result": (
                None
                if stage_one_decision is None
                else stage_one_decision["triggering_rule_id"]
            ),
            "policy_incident_status": policy_eligibility[
                "policy_incident_status"
            ],
            "plan_id": plan_id,
            "claim_category": claim_category,
            "coverage_lookup_status": coverage[
                "coverage_lookup_status"
            ],
            "covered_flag": coverage["covered_flag"],
            "annual_claims_used": annual_claims_used,
            "annual_claim_limit": annual_claim_limit,
            "annual_limit_exhausted": annual_limit_exhausted,
            "theft_claims_used": theft_claims_used,
            "maximum_theft_claims": maximum_theft_claims,
            "theft_limit_applicable": theft_limit_applicable,
            "theft_limit_exhausted": theft_limit_exhausted,
        }
    )

stage_two_audit_df = pd.DataFrame(stage_two_audit_rows)

print("Stage 2 hard-failure condition counts:")
display(
    pd.DataFrame(
        {
            "condition": [
                "Outside policy coverage period",
                "Explicitly uncovered category",
                "Annual limit exhausted",
                "Theft limit exhausted",
            ],
            "claim_count": [
                (
                    stage_two_audit_df["policy_incident_status"]
                    == "OUTSIDE_COVERAGE_PERIOD"
                ).sum(),
                (
                    stage_two_audit_df["covered_flag"] == False
                ).sum(),
                stage_two_audit_df["annual_limit_exhausted"].sum(),
                stage_two_audit_df["theft_limit_exhausted"].sum(),
            ],
        }
    )
)

stage_two_columns = [
    "claim_id",
    "stage_one_result",
    "policy_incident_status",
    "plan_id",
    "claim_category",
    "coverage_lookup_status",
    "covered_flag",
    "annual_claims_used",
    "annual_claim_limit",
    "annual_limit_exhausted",
    "theft_claims_used",
    "maximum_theft_claims",
    "theft_limit_exhausted",
]

print("\nPotential annual-limit cases:")
display(
    stage_two_audit_df.loc[
        stage_two_audit_df["annual_limit_exhausted"],
        stage_two_columns,
    ].sort_values("claim_id")
)

print("\nPotential theft-limit cases:")
display(
    stage_two_audit_df.loc[
        stage_two_audit_df["theft_limit_exhausted"],
        stage_two_columns,
    ].sort_values("claim_id")
)

print("\nAll hard-failure candidates that reached Stage 2:")
display(
    stage_two_audit_df.loc[
        (stage_two_audit_df["stage_one_result"].isna())
        & (
            (
                stage_two_audit_df["policy_incident_status"]
                == "OUTSIDE_COVERAGE_PERIOD"
            )
            | (stage_two_audit_df["covered_flag"] == False)
            | stage_two_audit_df["annual_limit_exhausted"]
            | stage_two_audit_df["theft_limit_exhausted"]
        ),
        stage_two_columns,
    ].sort_values("claim_id")
)

Stage 2 hard-failure condition counts:


,condition,claim_count
0,Outside policy coverage period,3
1,Explicitly uncovered category,15
2,Annual limit exhausted,7
3,Theft limit exhausted,2



Potential annual-limit cases:


,claim_id,stage_one_result,policy_incident_status,plan_id,claim_category,coverage_lookup_status,covered_flag,annual_claims_used,annual_claim_limit,annual_limit_exhausted,theft_claims_used,maximum_theft_claims,theft_limit_exhausted
139,CLM-000186,NaN,ACTIVE_ON_INCIDENT_DATE,DP-PREMIUM,LOSS,UNIQUE_COVERAGE_RECORD,False,2,2.0,True,0,1.0,False
150,CLM-000199,NaN,OUTSIDE_COVERAGE_PERIOD,DP-ESSENTIAL,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,1,1.0,True,0,0.0,False
153,CLM-000204,NaN,ACTIVE_ON_INCIDENT_DATE,DP-COMPLETE,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,2,2.0,True,0,0.0,False
154,CLM-000205,NaN,ACTIVE_ON_INCIDENT_DATE,DP-ESSENTIAL,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,1,1.0,True,0,0.0,False
155,CLM-000206,NaN,ACTIVE_ON_INCIDENT_DATE,DP-COMPLETE,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,2,2.0,True,0,0.0,False
156,CLM-000207,NaN,ACTIVE_ON_INCIDENT_DATE,DP-ESSENTIAL,SCREEN_DAMAGE,UNIQUE_COVERAGE_RECORD,True,1,1.0,True,0,0.0,False
157,CLM-000208,NaN,ACTIVE_ON_INCIDENT_DATE,DP-PREMIUM,LIQUID_DAMAGE,UNIQUE_COVERAGE_RECORD,True,2,2.0,True,0,1.0,False



Potential theft-limit cases:


,claim_id,stage_one_result,policy_incident_status,plan_id,claim_category,coverage_lookup_status,covered_flag,annual_claims_used,annual_claim_limit,annual_limit_exhausted,theft_claims_used,maximum_theft_claims,theft_limit_exhausted
158,CLM-000210,NaN,ACTIVE_ON_INCIDENT_DATE,DP-PREMIUM,THEFT,UNIQUE_COVERAGE_RECORD,True,1,2.0,False,1,1.0,True
159,CLM-000211,NaN,ACTIVE_ON_INCIDENT_DATE,DP-PREMIUM,THEFT,UNIQUE_COVERAGE_RECORD,True,1,2.0,False,1,1.0,True



All hard-failure candidates that reached Stage 2:


,claim_id,stage_one_result,policy_incident_status,plan_id,claim_category,coverage_lookup_status,covered_flag,annual_claims_used,annual_claim_limit,annual_limit_exhausted,theft_claims_used,maximum_theft_claims,theft_limit_exhausted
116,CLM-000154,NaN,ACTIVE_ON_INCIDENT_DATE,DP-ESSENTIAL,THEFT,UNIQUE_COVERAGE_RECORD,False,0,1.0,False,0,0.0,False
118,CLM-000156,NaN,ACTIVE_ON_INCIDENT_DATE,DP-ESSENTIAL,THEFT,UNIQUE_COVERAGE_RECORD,False,0,1.0,False,0,0.0,False
119,CLM-000157,NaN,ACTIVE_ON_INCIDENT_DATE,DP-COMPLETE,THEFT,UNIQUE_COVERAGE_RECORD,False,0,2.0,False,0,0.0,False
139,CLM-000186,NaN,ACTIVE_ON_INCIDENT_DATE,DP-PREMIUM,LOSS,UNIQUE_COVERAGE_RECORD,False,2,2.0,True,0,1.0,False
140,CLM-000187,NaN,ACTIVE_ON_INCIDENT_DATE,DP-PREMIUM,LOSS,UNIQUE_COVERAGE_RECORD,False,0,2.0,False,0,1.0,False
141,CLM-000188,NaN,ACTIVE_ON_INCIDENT_DATE,DP-PREMIUM,LOSS,UNIQUE_COVERAGE_RECORD,False,0,2.0,False,0,1.0,False
142,CLM-000189,NaN,ACTIVE_ON_INCIDENT_DATE,DP-PREMIUM,LOSS,UNIQUE_COVERAGE_RECORD,False,0,2.0,False,0,1.0,False
143,CLM-000190,NaN,ACTIVE_ON_INCIDENT_DATE,DP-COMPLETE,THEFT,UNIQUE_COVERAGE_RECORD,False,0,2.0,False,0,0.0,False
144,CLM-000191,NaN,ACTIVE_ON_INCIDENT_DATE,DP-ESSENTIAL,THEFT,UNIQUE_COVERAGE_RECORD,False,0,1.0,False,0,0.0,False
145,CLM-000192,NaN,ACTIVE_ON_INCIDENT_DATE,DP-COMPLETE,LOSS,UNIQUE_COVERAGE_RECORD,False,0,2.0,False,0,0.0,False


In [69]:
# ============================================================
# Deterministic Triage Engine — Step 12: Stage 2 Validation
# ============================================================

import importlib

import src.triage_engine as triage_engine_module

importlib.reload(triage_engine_module)

evaluate_stage_one = triage_engine_module.evaluate_stage_one
evaluate_stage_two = triage_engine_module.evaluate_stage_two


def evaluate_through_stage_two(context: dict) -> dict | None:
    """Run Stage 1, then Stage 2 only when Stage 1 has no decision."""
    shared_trace = []

    stage_one_result = evaluate_stage_one(
        context,
        rule_trace=shared_trace,
    )

    if stage_one_result is not None:
        return stage_one_result

    return evaluate_stage_two(
        context,
        rule_trace=shared_trace,
    )


stage_two_ready = stage_two_audit_df[
    stage_two_audit_df["stage_one_result"].isna()
].copy()

outside_coverage_claim_id = stage_two_ready.loc[
    stage_two_ready["policy_incident_status"]
    == "OUTSIDE_COVERAGE_PERIOD",
    "claim_id",
].iloc[0]

explicitly_uncovered_claim_id = stage_two_ready.loc[
    stage_two_ready["covered_flag"] == False,
    "claim_id",
].iloc[0]

annual_limit_claim_id = stage_two_ready.loc[
    (stage_two_ready["annual_limit_exhausted"])
    & (stage_two_ready["covered_flag"] == True)
    & (
        stage_two_ready["policy_incident_status"]
        == "ACTIVE_ON_INCIDENT_DATE"
    ),
    "claim_id",
].iloc[0]

theft_limit_claim_id = stage_two_ready.loc[
    stage_two_ready["theft_limit_exhausted"],
    "claim_id",
].iloc[0]

uncovered_and_annual_limit_claim_id = stage_two_ready.loc[
    (stage_two_ready["covered_flag"] == False)
    & (stage_two_ready["annual_limit_exhausted"]),
    "claim_id",
].iloc[0]

outside_coverage_and_annual_limit_claim_id = stage_two_ready.loc[
    (
        stage_two_ready["policy_incident_status"]
        == "OUTSIDE_COVERAGE_PERIOD"
    )
    & (stage_two_ready["annual_limit_exhausted"]),
    "claim_id",
].iloc[0]

scenario_claim_ids = {
    "Clean claim": "CLM-000001",
    "Outside policy coverage period": outside_coverage_claim_id,
    "Explicitly uncovered category": explicitly_uncovered_claim_id,
    "Covered annual limit exhausted": annual_limit_claim_id,
    "Premium theft limit exhausted": theft_limit_claim_id,
    "Uncovered category plus annual limit exhausted": (
        uncovered_and_annual_limit_claim_id
    ),
    "Outside coverage plus annual limit exhausted": (
        outside_coverage_and_annual_limit_claim_id
    ),
}

stage_two_results = []

for scenario_name, claim_id in scenario_claim_ids.items():
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    result = evaluate_through_stage_two(context)

    stage_two_results.append(
        {
            "scenario": scenario_name,
            "claim_id": claim_id,
            "triage_outcome": (
                None if result is None else result["triage_outcome"]
            ),
            "triggering_rule_id": (
                None if result is None else result["triggering_rule_id"]
            ),
            "precedence_stage": (
                None if result is None else result["precedence_stage"]
            ),
            "decision_reason": (
                "No decision through Stage 2; continue to Stage 3."
                if result is None
                else result["decision_reason"]
            ),
        }
    )

stage_two_results_df = pd.DataFrame(stage_two_results)

display(stage_two_results_df)

print("Rule trace for the combined uncovered + annual-limit claim:")
combined_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id=uncovered_and_annual_limit_claim_id,
)

combined_result = evaluate_through_stage_two(combined_context)

for trace_item in combined_result["rule_trace"]:
    print(trace_item)

,scenario,claim_id,triage_outcome,triggering_rule_id,precedence_stage,decision_reason
0,Clean claim,CLM-000001,NaN,NaN,NaN,No decision through Stage 2; continue to Stage 3.
1,Outside policy coverage period,CLM-000196,NOT_ELIGIBLE,ELG-002,2.0,The policy was not active on the reported inci...
2,Explicitly uncovered category,CLM-000154,NOT_ELIGIBLE,COV-001,2.0,The identified claim category is explicitly no...
3,Covered annual limit exhausted,CLM-000204,NOT_ELIGIBLE,LIM-001,2.0,The plan's annual claim allowance is exhausted.
4,Premium theft limit exhausted,CLM-000210,NOT_ELIGIBLE,LIM-002,2.0,The plan's theft claim allowance is exhausted.
5,Uncovered category plus annual limit exhausted,CLM-000186,NOT_ELIGIBLE,COV-001,2.0,The identified claim category is explicitly no...
6,Outside coverage plus annual limit exhausted,CLM-000199,NOT_ELIGIBLE,ELG-002,2.0,The policy was not active on the reported inci...


Rule trace for the combined uncovered + annual-limit claim:
{'rule_id': 'DATA-001', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'DATA-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'ELG-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'ACTIVE_ON_INCIDENT_DATE'}
{'rule_id': 'DATA-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': {'plan_configuration_status': 'ACTIVE_CONFIGURATION_AVAILABLE', 'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD'}}
{'rule_id': 'ELG-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'IN_SCOPE'}
{'rule_id': 'DEV-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'DEVICE_MATCH'}
{'rule_id': 'EVD-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'NOT_APPLICABLE'}
{'rule_id': 'ELG-002', 'precedence_stage': 2, 'eva

In [70]:
# ============================================================
# Deterministic Triage Engine — Step 13: Stage 3 Missing-Fact Audit
# ============================================================

stage_three_audit_rows = []

for claim_id in data["development_claims"]["claim_id"]:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    shared_trace = []

    stage_one_result = evaluate_stage_one(
        context,
        rule_trace=shared_trace,
    )

    if stage_one_result is not None:
        stage_two_result = None
    else:
        stage_two_result = evaluate_stage_two(
            context,
            rule_trace=shared_trace,
        )

    claim = context["claim"]
    policy_eligibility = context["policy_eligibility"]
    device_match = context["device_match"]

    claim_category = claim["claim_category_selected"]

    claim_category_missing = (
        pd.isna(claim_category)
        or str(claim_category).strip() == ""
        or str(claim_category).strip().upper() == "UNSPECIFIED"
    )

    incident_date_missing_or_invalid = (
        policy_eligibility["policy_incident_status"]
        == "INCIDENT_DATE_MISSING_OR_INVALID"
    )

    device_identifier_missing = (
        device_match["status"] == "DEVICE_IDENTIFIER_MISSING"
    )

    stage_three_audit_rows.append(
        {
            "claim_id": claim_id,
            "stage_one_result": (
                None
                if stage_one_result is None
                else stage_one_result["triggering_rule_id"]
            ),
            "stage_two_result": (
                None
                if stage_two_result is None
                else stage_two_result["triggering_rule_id"]
            ),
            "reported_incident_date": claim["reported_incident_date"],
            "claim_category_selected": claim_category,
            "claimed_device_identifier": claim[
                "claimed_device_identifier"
            ],
            "incident_date_missing_or_invalid": (
                incident_date_missing_or_invalid
            ),
            "claim_category_missing": claim_category_missing,
            "device_identifier_missing": device_identifier_missing,
        }
    )

stage_three_audit_df = pd.DataFrame(stage_three_audit_rows)

stage_three_ready = stage_three_audit_df[
    stage_three_audit_df["stage_one_result"].isna()
    & stage_three_audit_df["stage_two_result"].isna()
].copy()

stage_three_ready["missing_fact_count"] = (
    stage_three_ready["incident_date_missing_or_invalid"].astype(int)
    + stage_three_ready["claim_category_missing"].astype(int)
    + stage_three_ready["device_identifier_missing"].astype(int)
)

print("Stage 3 missing-fact counts for claims reaching Stage 3:")
display(
    pd.DataFrame(
        {
            "missing_fact_type": [
                "Missing or invalid incident date",
                "Missing or UNSPECIFIED claim category",
                "Missing device identifier",
            ],
            "claim_count": [
                stage_three_ready[
                    "incident_date_missing_or_invalid"
                ].sum(),
                stage_three_ready["claim_category_missing"].sum(),
                stage_three_ready["device_identifier_missing"].sum(),
            ],
        }
    )
)

print("\nClaims with one or more Stage 3 missing facts:")
display(
    stage_three_ready.loc[
        stage_three_ready["missing_fact_count"] > 0,
        [
            "claim_id",
            "reported_incident_date",
            "claim_category_selected",
            "claimed_device_identifier",
            "incident_date_missing_or_invalid",
            "claim_category_missing",
            "device_identifier_missing",
            "missing_fact_count",
        ],
    ].sort_values(["missing_fact_count", "claim_id"], ascending=[False, True])
)

print("\nClaims with more than one missing fact:")
display(
    stage_three_ready.loc[
        stage_three_ready["missing_fact_count"] > 1,
        [
            "claim_id",
            "incident_date_missing_or_invalid",
            "claim_category_missing",
            "device_identifier_missing",
            "missing_fact_count",
        ],
    ].sort_values("claim_id")
)

Stage 3 missing-fact counts for claims reaching Stage 3:


,missing_fact_type,claim_count
0,Missing or invalid incident date,11
1,Missing or UNSPECIFIED claim category,9
2,Missing device identifier,14



Claims with one or more Stage 3 missing facts:


,claim_id,reported_incident_date,claim_category_selected,claimed_device_identifier,incident_date_missing_or_invalid,claim_category_missing,device_identifier_missing,missing_fact_count
53,CLM-000071,NaN,MECHANICAL_BREAKDOWN,DVC-00000045,True,False,False,1
54,CLM-000072,NaN,MECHANICAL_BREAKDOWN,DVC-00000100,True,False,False,1
55,CLM-000073,NaN,SCREEN_DAMAGE,DVC-00000021,True,False,False,1
56,CLM-000074,NaN,SCREEN_DAMAGE,DVC-00000037,True,False,False,1
57,CLM-000075,NaN,LIQUID_DAMAGE,DVC-00000105,True,False,False,1
58,CLM-000076,NaN,MECHANICAL_BREAKDOWN,DVC-00000034,True,False,False,1
59,CLM-000077,NaN,SCREEN_DAMAGE,DVC-00000085,True,False,False,1
60,CLM-000078,NaN,SCREEN_DAMAGE,DVC-00000090,True,False,False,1
61,CLM-000079,NaN,ACCIDENTAL_DAMAGE,DVC-00000056,True,False,False,1
62,CLM-000080,NaN,SCREEN_DAMAGE,DVC-00000037,True,False,False,1



Claims with more than one missing fact:


,claim_id,incident_date_missing_or_invalid,claim_category_missing,device_identifier_missing,missing_fact_count


In [71]:
# ============================================================
# Deterministic Triage Engine — Step 15: Stage 3 Validation
# ============================================================

import importlib

import src.triage_engine as triage_engine_module

importlib.reload(triage_engine_module)

evaluate_stage_one = triage_engine_module.evaluate_stage_one
evaluate_stage_two = triage_engine_module.evaluate_stage_two
evaluate_stage_three = triage_engine_module.evaluate_stage_three


def evaluate_through_stage_three(context: dict) -> dict | None:
    """Run Stages 1, 2, and 3 in order."""
    shared_trace = []

    stage_one_result = evaluate_stage_one(
        context,
        rule_trace=shared_trace,
    )

    if stage_one_result is not None:
        return stage_one_result

    stage_two_result = evaluate_stage_two(
        context,
        rule_trace=shared_trace,
    )

    if stage_two_result is not None:
        return stage_two_result

    return evaluate_stage_three(
        context,
        rule_trace=shared_trace,
    )


stage_three_ready = stage_three_audit_df[
    stage_three_audit_df["stage_one_result"].isna()
    & stage_three_audit_df["stage_two_result"].isna()
].copy()

missing_incident_date_claim_id = stage_three_ready.loc[
    stage_three_ready["incident_date_missing_or_invalid"],
    "claim_id",
].iloc[0]

unspecified_category_claim_id = stage_three_ready.loc[
    stage_three_ready["claim_category_missing"],
    "claim_id",
].iloc[0]

missing_device_identifier_claim_id = stage_three_ready.loc[
    stage_three_ready["device_identifier_missing"],
    "claim_id",
].iloc[0]

scenario_claim_ids = {
    "Clean claim": "CLM-000001",
    "Missing incident date": missing_incident_date_claim_id,
    "Unspecified claim category": unspecified_category_claim_id,
    "Missing device identifier": missing_device_identifier_claim_id,
}

stage_three_results = []

for scenario_name, claim_id in scenario_claim_ids.items():
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    result = evaluate_through_stage_three(context)

    stage_three_results.append(
        {
            "scenario": scenario_name,
            "claim_id": claim_id,
            "triage_outcome": (
                None if result is None else result["triage_outcome"]
            ),
            "triggering_rule_id": (
                None if result is None else result["triggering_rule_id"]
            ),
            "precedence_stage": (
                None if result is None else result["precedence_stage"]
            ),
            "decision_reason": (
                "No decision through Stage 3; continue to Stage 4."
                if result is None
                else result["decision_reason"]
            ),
        }
    )

stage_three_results_df = pd.DataFrame(stage_three_results)

display(stage_three_results_df)

print("Rule trace for missing incident date:")
missing_date_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id=missing_incident_date_claim_id,
)

missing_date_result = evaluate_through_stage_three(missing_date_context)

for trace_item in missing_date_result["rule_trace"]:
    print(trace_item)

,scenario,claim_id,triage_outcome,triggering_rule_id,precedence_stage,decision_reason
0,Clean claim,CLM-000001,NaN,NaN,NaN,No decision through Stage 3; continue to Stage 4.
1,Missing incident date,CLM-000071,INFO_REQUIRED,ELG-001,3.0,A valid incident date is required before incid...
2,Unspecified claim category,CLM-000086,INFO_REQUIRED,CLM-001,3.0,The claim category is missing or unspecified. ...
3,Missing device identifier,CLM-000098,INFO_REQUIRED,DEV-001,3.0,"A usable device identifier, such as IMEI or se..."


Rule trace for missing incident date:
{'rule_id': 'DATA-001', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'DATA-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'ELG-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'INCIDENT_DATE_MISSING_OR_INVALID'}
{'rule_id': 'DATA-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': {'plan_configuration_status': 'INCIDENT_DATE_MISSING_OR_INVALID', 'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD'}}
{'rule_id': 'ELG-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'NOT_ASSESSED'}
{'rule_id': 'DEV-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'DEVICE_MATCH'}
{'rule_id': 'EVD-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'SUFFICIENT'}
{'rule_id': 'ELG-002', 'precedence_stage': 2, 'evaluation': '

In [72]:
# ============================================================
# Deterministic Triage Engine — Step 16: Risk Signal Audit
# ============================================================

from pprint import pprint

risk_claim_ids = context_status_audit.loc[
    context_status_audit["has_triggered_risk"] == True,
    "claim_id",
].tolist()

print("Risk-triggered claim IDs:")
print(risk_claim_ids)

for claim_id in risk_claim_ids:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    print(f"\n{'=' * 80}")
    print("Claim ID:", claim_id)
    print("Claim category:", context["claim"]["claim_category_selected"])
    print("Policy lookup status:", context["policy_lookup_status"])
    print(
        "Policy incident status:",
        context["policy_eligibility"]["policy_incident_status"],
    )
    print("Coverage flag:", context["coverage"]["covered_flag"])
    print(
        "Evidence status:",
        context["evidence"]["evidence_assessment_status"],
    )

    print("\nRisk assessment:")
    pprint(context["risk"])

Risk-triggered claim IDs:
['CLM-000180', 'CLM-000181', 'CLM-000183', 'CLM-000184']

Claim ID: CLM-000180
Claim category: MECHANICAL_BREAKDOWN
Policy lookup status: UNIQUE_MATCH
Policy incident status: ACTIVE_ON_INCIDENT_DATE
Coverage flag: True
Evidence status: SUFFICIENT

Risk assessment:
{'has_triggered_risk': True,
 'manual_review_reasons': ['POTENTIAL_DUPLICATE'],
 'recommended_action': 'MANUAL_REVIEW',
 'risk_indicator_ids': ['RSK-002'],
 'risk_indicator_names': ['RECENT_RELATED_CLAIM_PATTERN']}

Claim ID: CLM-000181
Claim category: SCREEN_DAMAGE
Policy lookup status: UNIQUE_MATCH
Policy incident status: ACTIVE_ON_INCIDENT_DATE
Coverage flag: True
Evidence status: SUFFICIENT

Risk assessment:
{'has_triggered_risk': True,
 'manual_review_reasons': ['POTENTIAL_DUPLICATE'],
 'recommended_action': 'MANUAL_REVIEW',
 'risk_indicator_ids': ['RSK-002'],
 'risk_indicator_names': ['RECENT_RELATED_CLAIM_PATTERN']}

Claim ID: CLM-000183
Claim category: MECHANICAL_BREAKDOWN
Policy lookup statu

In [73]:
# ============================================================
# Deterministic Triage Engine — Step 18: Stage 4 Validation
# ============================================================

import importlib

import src.triage_engine as triage_engine_module

importlib.reload(triage_engine_module)

evaluate_stage_one = triage_engine_module.evaluate_stage_one
evaluate_stage_two = triage_engine_module.evaluate_stage_two
evaluate_stage_three = triage_engine_module.evaluate_stage_three
evaluate_stage_four = triage_engine_module.evaluate_stage_four


def evaluate_through_stage_four(context: dict) -> dict | None:
    """Run Stages 1 through 4 in precedence order."""
    shared_trace = []

    for evaluator in [
        evaluate_stage_one,
        evaluate_stage_two,
        evaluate_stage_three,
        evaluate_stage_four,
    ]:
        result = evaluator(
            context,
            rule_trace=shared_trace,
        )

        if result is not None:
            return result

    return None


stage_four_scenarios = {
    "Clean claim": "CLM-000001",
}

for claim_id in risk_claim_ids:
    stage_four_scenarios[f"Risk-triggered claim: {claim_id}"] = claim_id

stage_four_results = []

for scenario_name, claim_id in stage_four_scenarios.items():
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    result = evaluate_through_stage_four(context)

    stage_four_results.append(
        {
            "scenario": scenario_name,
            "claim_id": claim_id,
            "triage_outcome": (
                None if result is None else result["triage_outcome"]
            ),
            "triggering_rule_id": (
                None if result is None else result["triggering_rule_id"]
            ),
            "precedence_stage": (
                None if result is None else result["precedence_stage"]
            ),
            "decision_reason": (
                "No decision through Stage 4; continue to Stage 5."
                if result is None
                else result["decision_reason"]
            ),
        }
    )

stage_four_results_df = pd.DataFrame(stage_four_results)

display(stage_four_results_df)

print("Rule trace for the first risk-triggered claim:")

first_risk_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id=risk_claim_ids[0],
)

first_risk_result = evaluate_through_stage_four(first_risk_context)

for trace_item in first_risk_result["rule_trace"]:
    print(trace_item)

,scenario,claim_id,triage_outcome,triggering_rule_id,precedence_stage,decision_reason
0,Clean claim,CLM-000001,NaN,NaN,NaN,No decision through Stage 4; continue to Stage 5.
1,Risk-triggered claim: CLM-000180,CLM-000180,MANUAL_REVIEW,ANM-001,4.0,A material risk or anomaly signal requires ana...
2,Risk-triggered claim: CLM-000181,CLM-000181,MANUAL_REVIEW,ANM-001,4.0,A material risk or anomaly signal requires ana...
3,Risk-triggered claim: CLM-000183,CLM-000183,MANUAL_REVIEW,ANM-001,4.0,A material risk or anomaly signal requires ana...
4,Risk-triggered claim: CLM-000184,CLM-000184,MANUAL_REVIEW,ANM-001,4.0,A material risk or anomaly signal requires ana...


Rule trace for the first risk-triggered claim:
{'rule_id': 'DATA-001', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'DATA-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'ELG-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'ACTIVE_ON_INCIDENT_DATE'}
{'rule_id': 'DATA-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': {'plan_configuration_status': 'ACTIVE_CONFIGURATION_AVAILABLE', 'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD'}}
{'rule_id': 'ELG-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'IN_SCOPE'}
{'rule_id': 'DEV-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'DEVICE_MATCH'}
{'rule_id': 'EVD-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'SUFFICIENT'}
{'rule_id': 'ELG-002', 'precedence_stage': 2, 'evaluation': 'NOT_TR

In [74]:
# ============================================================
# Deterministic Triage Engine — Step 19: Evidence Gap Audit
# ============================================================

evidence_gap_rows = []

for claim_id in data["development_claims"]["claim_id"]:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    shared_trace = []

    stage_one_result = evaluate_stage_one(
        context,
        rule_trace=shared_trace,
    )

    stage_two_result = None
    stage_three_result = None
    stage_four_result = None

    if stage_one_result is None:
        stage_two_result = evaluate_stage_two(
            context,
            rule_trace=shared_trace,
        )

    if stage_one_result is None and stage_two_result is None:
        stage_three_result = evaluate_stage_three(
            context,
            rule_trace=shared_trace,
        )

    if (
        stage_one_result is None
        and stage_two_result is None
        and stage_three_result is None
    ):
        stage_four_result = evaluate_stage_four(
            context,
            rule_trace=shared_trace,
        )

    evidence = context["evidence"]

    evidence_gap_rows.append(
        {
            "claim_id": claim_id,
            "stage_one_result": (
                None
                if stage_one_result is None
                else stage_one_result["triggering_rule_id"]
            ),
            "stage_two_result": (
                None
                if stage_two_result is None
                else stage_two_result["triggering_rule_id"]
            ),
            "stage_three_result": (
                None
                if stage_three_result is None
                else stage_three_result["triggering_rule_id"]
            ),
            "stage_four_result": (
                None
                if stage_four_result is None
                else stage_four_result["triggering_rule_id"]
            ),
            "evidence_assessment_status": evidence[
                "evidence_assessment_status"
            ],
            "evidence_profile_id": evidence.get("evidence_profile_id"),
            "missing_required_items": evidence.get(
                "missing_required_items"
            ),
            "unreadable_required_items": evidence.get(
                "unreadable_required_items"
            ),
        }
    )

evidence_gap_audit_df = pd.DataFrame(evidence_gap_rows)

stage_five_ready = evidence_gap_audit_df[
    evidence_gap_audit_df["stage_one_result"].isna()
    & evidence_gap_audit_df["stage_two_result"].isna()
    & evidence_gap_audit_df["stage_three_result"].isna()
    & evidence_gap_audit_df["stage_four_result"].isna()
].copy()

print("Incomplete evidence cases reaching Stage 5:")
display(
    stage_five_ready.loc[
        stage_five_ready["evidence_assessment_status"] == "INCOMPLETE"
    ].sort_values("claim_id")
)

incomplete_evidence_claim_ids = stage_five_ready.loc[
    stage_five_ready["evidence_assessment_status"] == "INCOMPLETE",
    "claim_id",
].tolist()

print("\nDetailed evidence assessment for each Stage 5 candidate:")

for claim_id in incomplete_evidence_claim_ids:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    print(f"\n{'=' * 80}")
    print("Claim ID:", claim_id)
    print("Claim category:", context["claim"]["claim_category_selected"])
    pprint(context["evidence"])

Incomplete evidence cases reaching Stage 5:


,claim_id,stage_one_result,stage_two_result,stage_three_result,stage_four_result,evidence_assessment_status,evidence_profile_id,missing_required_items,unreadable_required_items
87,CLM-000117,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-LIQUID-01,None,None
88,CLM-000118,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-MECHANICAL-01,None,None
89,CLM-000119,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-LIQUID-01,None,None
90,CLM-000120,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-SCREEN-01,None,None
91,CLM-000121,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-SCREEN-01,None,None
92,CLM-000122,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-MECHANICAL-01,None,None
93,CLM-000123,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-SCREEN-01,None,None
94,CLM-000124,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-MECHANICAL-01,None,None
95,CLM-000125,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-ACCIDENTAL-01,None,None
96,CLM-000126,NaN,NaN,NaN,NaN,INCOMPLETE,EVD-LIQUID-01,None,None



Detailed evidence assessment for each Stage 5 candidate:

Claim ID: CLM-000117
Claim category: LIQUID_DAMAGE
{'evidence_assessment_status': 'INCOMPLETE',
 'evidence_profile_id': 'EVD-LIQUID-01',
 'has_contradictory_evidence': False,
 'missing_required_evidence_types': ['DAMAGE_PHOTO'],
 'required_evidence_types': ['DAMAGE_PHOTO'],
 'submitted_evidence_types': [],
 'unreadable_required_evidence_types': []}

Claim ID: CLM-000118
Claim category: MECHANICAL_BREAKDOWN
{'evidence_assessment_status': 'INCOMPLETE',
 'evidence_profile_id': 'EVD-MECHANICAL-01',
 'has_contradictory_evidence': False,
 'missing_required_evidence_types': ['DIAGNOSTIC_REPORT'],
 'required_evidence_types': ['DIAGNOSTIC_REPORT'],
 'submitted_evidence_types': ['DAMAGE_PHOTO'],
 'unreadable_required_evidence_types': []}

Claim ID: CLM-000119
Claim category: LIQUID_DAMAGE
{'evidence_assessment_status': 'INCOMPLETE',
 'evidence_profile_id': 'EVD-LIQUID-01',
 'has_contradictory_evidence': False,
 'missing_required_evidence

In [75]:
# ============================================================
# Deterministic Triage Engine — Step 21: Stage 5 Validation
# ============================================================

import importlib

import src.triage_engine as triage_engine_module

importlib.reload(triage_engine_module)

evaluate_stage_one = triage_engine_module.evaluate_stage_one
evaluate_stage_two = triage_engine_module.evaluate_stage_two
evaluate_stage_three = triage_engine_module.evaluate_stage_three
evaluate_stage_four = triage_engine_module.evaluate_stage_four
evaluate_stage_five = triage_engine_module.evaluate_stage_five


def evaluate_through_stage_five(context: dict) -> dict | None:
    """Run Stages 1 through 5 in precedence order."""
    shared_trace = []

    for evaluator in [
        evaluate_stage_one,
        evaluate_stage_two,
        evaluate_stage_three,
        evaluate_stage_four,
        evaluate_stage_five,
    ]:
        result = evaluator(
            context,
            rule_trace=shared_trace,
        )

        if result is not None:
            return result

    return None


missing_evidence_claim_id = incomplete_evidence_claim_ids[0]

unreadable_evidence_claim_id = next(
    claim_id
    for claim_id in incomplete_evidence_claim_ids
    if claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )["evidence"]["unreadable_required_evidence_types"]
)

stage_five_scenarios = {
    "Clean claim": "CLM-000001",
    "Missing required evidence": missing_evidence_claim_id,
    "Unreadable required evidence": unreadable_evidence_claim_id,
}

stage_five_results = []

for scenario_name, claim_id in stage_five_scenarios.items():
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    result = evaluate_through_stage_five(context)

    stage_five_results.append(
        {
            "scenario": scenario_name,
            "claim_id": claim_id,
            "triage_outcome": (
                None if result is None else result["triage_outcome"]
            ),
            "triggering_rule_id": (
                None if result is None else result["triggering_rule_id"]
            ),
            "precedence_stage": (
                None if result is None else result["precedence_stage"]
            ),
            "decision_reason": (
                "No decision through Stage 5; continue to OUT-001."
                if result is None
                else result["decision_reason"]
            ),
        }
    )

stage_five_results_df = pd.DataFrame(stage_five_results)

display(stage_five_results_df)

print("Rule trace for unreadable required evidence:")

unreadable_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id=unreadable_evidence_claim_id,
)

unreadable_result = evaluate_through_stage_five(unreadable_context)

for trace_item in unreadable_result["rule_trace"]:
    print(trace_item)

,scenario,claim_id,triage_outcome,triggering_rule_id,precedence_stage,decision_reason
0,Clean claim,CLM-000001,NaN,NaN,NaN,No decision through Stage 5; continue to OUT-001.
1,Missing required evidence,CLM-000117,INFO_REQUIRED,EVD-001,5.0,Required evidence is missing or unreadable. Re...
2,Unreadable required evidence,CLM-000120,INFO_REQUIRED,EVD-001,5.0,Required evidence is missing or unreadable. Re...


Rule trace for unreadable required evidence:
{'rule_id': 'DATA-001', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'DATA-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'UNIQUE_MATCH'}
{'rule_id': 'ELG-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'ACTIVE_ON_INCIDENT_DATE'}
{'rule_id': 'DATA-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': {'plan_configuration_status': 'ACTIVE_CONFIGURATION_AVAILABLE', 'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD'}}
{'rule_id': 'ELG-003', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'IN_SCOPE'}
{'rule_id': 'DEV-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'DEVICE_MATCH'}
{'rule_id': 'EVD-002', 'precedence_stage': 1, 'evaluation': 'NOT_TRIGGERED', 'observed_value': 'INCOMPLETE'}
{'rule_id': 'ELG-002', 'precedence_stage': 2, 'evaluation': 'NOT_TRIG

In [76]:
# ============================================================
# Deterministic Triage Engine — Step 23: Full Engine Validation
# ============================================================

import importlib

import src.triage_engine as triage_engine_module

importlib.reload(triage_engine_module)

triage_claim = triage_engine_module.triage_claim

full_engine_scenarios = {
    "Clean claim": "CLM-000001",
    "Multiple policy match": multiple_match_claim_id,
    "Outside policy coverage period": outside_coverage_claim_id,
    "Missing incident date": missing_incident_date_claim_id,
    "Risk-triggered claim": risk_claim_ids[0],
    "Missing required evidence": missing_evidence_claim_id,
    "Unreadable required evidence": unreadable_evidence_claim_id,
    "Explicitly uncovered category": explicitly_uncovered_claim_id,
    "Annual limit exhausted": annual_limit_claim_id,
    "Premium theft limit exhausted": theft_limit_claim_id,
}

full_engine_results = []

for scenario_name, claim_id in full_engine_scenarios.items():
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    decision = triage_claim(context)

    full_engine_results.append(
        {
            "scenario": scenario_name,
            "claim_id": claim_id,
            "triage_outcome": decision["triage_outcome"],
            "triggering_rule_id": decision["triggering_rule_id"],
            "precedence_stage": decision["precedence_stage"],
            "decision_reason": decision["decision_reason"],
            "rule_trace_length": len(decision["rule_trace"]),
        }
    )

full_engine_results_df = pd.DataFrame(full_engine_results)

display(full_engine_results_df)

print("Full decision for the clean claim:")
clean_context = claim_context_module.assemble_claim_context(
    data=data,
    claim_id="CLM-000001",
)

clean_decision = triage_claim(clean_context)

pprint(clean_decision)

,scenario,claim_id,triage_outcome,triggering_rule_id,precedence_stage,decision_reason,rule_trace_length
0,Clean claim,CLM-000001,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17
1,Multiple policy match,CLM-000162,MANUAL_REVIEW,DATA-002,1,Multiple candidate policy records were found. ...,2
2,Outside policy coverage period,CLM-000196,NOT_ELIGIBLE,ELG-002,2,The policy was not active on the reported inci...,8
3,Missing incident date,CLM-000071,INFO_REQUIRED,ELG-001,3,A valid incident date is required before incid...,12
4,Risk-triggered claim,CLM-000180,MANUAL_REVIEW,ANM-001,4,A material risk or anomaly signal requires ana...,15
5,Missing required evidence,CLM-000117,INFO_REQUIRED,EVD-001,5,Required evidence is missing or unreadable. Re...,16
6,Unreadable required evidence,CLM-000120,INFO_REQUIRED,EVD-001,5,Required evidence is missing or unreadable. Re...,16
7,Explicitly uncovered category,CLM-000154,NOT_ELIGIBLE,COV-001,2,The identified claim category is explicitly no...,9
8,Annual limit exhausted,CLM-000204,NOT_ELIGIBLE,LIM-001,2,The plan's annual claim allowance is exhausted.,10
9,Premium theft limit exhausted,CLM-000210,NOT_ELIGIBLE,LIM-002,2,The plan's theft claim allowance is exhausted.,11


Full decision for the clean claim:
{'claim_id': 'CLM-000001',
 'decision_reason': 'All applicable deterministic checks passed. Eligible for '
                    'standard processing only; this is not an approval, '
                    'payout, fraud determination, or final denial decision.',
 'decision_support_only': True,
 'precedence_stage': 6,
 'rule_trace': [{'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'UNIQUE_MATCH',
                 'precedence_stage': 1,
                 'rule_id': 'DATA-001'},
                {'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'UNIQUE_MATCH',
                 'precedence_stage': 1,
                 'rule_id': 'DATA-002'},
                {'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'ACTIVE_ON_INCIDENT_DATE',
                 'precedence_stage': 1,
                 'rule_id': 'ELG-002'},
                {'evaluation': 'NOT_TRIGGERED',
                 'observed_value': {'coverage_lookup

In [77]:
# ============================================================
# Deterministic Triage Engine — Step 24: Full Batch Reconciliation
# ============================================================

batch_decisions = []

for claim_id in data["development_claims"]["claim_id"]:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    decision = triage_claim(context)

    batch_decisions.append(
        {
            "claim_id": claim_id,
            "triage_outcome": decision["triage_outcome"],
            "triggering_rule_id": decision["triggering_rule_id"],
            "precedence_stage": decision["precedence_stage"],
            "decision_reason": decision["decision_reason"],
            "rule_trace_length": len(decision["rule_trace"]),
            "decision_support_only": decision["decision_support_only"],
        }
    )

batch_decisions_df = pd.DataFrame(batch_decisions)

allowed_outcomes = {
    "PROCEED",
    "INFO_REQUIRED",
    "MANUAL_REVIEW",
    "NOT_ELIGIBLE",
}

print("Batch validation checks:")
print("Development claims:", len(data["development_claims"]))
print("Decisions returned:", len(batch_decisions_df))
print(
    "All outcomes valid:",
    batch_decisions_df["triage_outcome"].isin(allowed_outcomes).all(),
)
print(
    "All decisions marked decision-support-only:",
    batch_decisions_df["decision_support_only"].all(),
)
print(
    "Any missing triggering rule:",
    batch_decisions_df["triggering_rule_id"].isna().any(),
)

print("\nOutcome distribution:")
display(
    batch_decisions_df[
        "triage_outcome"
    ].value_counts().rename_axis("triage_outcome").reset_index(name="claim_count")
)

print("\nTriggering-rule distribution:")
display(
    batch_decisions_df[
        "triggering_rule_id"
    ].value_counts().rename_axis("triggering_rule_id").reset_index(name="claim_count")
)

print("\nOutcome and triggering-rule distribution:")
display(
    batch_decisions_df.groupby(
        ["triage_outcome", "triggering_rule_id"],
        as_index=False,
    ).size().rename(columns={"size": "claim_count"}).sort_values(
        ["triage_outcome", "triggering_rule_id"]
    )
)

print("\nClaims routed to PROCEED:")
display(
    batch_decisions_df.loc[
        batch_decisions_df["triage_outcome"] == "PROCEED",
        [
            "claim_id",
            "triage_outcome",
            "triggering_rule_id",
            "precedence_stage",
            "rule_trace_length",
        ],
    ].head(10)
)

print("\nClaims with the longest audit traces:")
display(
    batch_decisions_df.sort_values(
        ["rule_trace_length", "claim_id"],
        ascending=[False, True],
    ).head(10)
)

Batch validation checks:
Development claims: 165
Decisions returned: 165
All outcomes valid: True
All decisions marked decision-support-only: True
Any missing triggering rule: False

Outcome distribution:


,triage_outcome,claim_count
0,PROCEED,64
1,INFO_REQUIRED,45
2,MANUAL_REVIEW,31
3,NOT_ELIGIBLE,25



Triggering-rule distribution:


,triggering_rule_id,claim_count
0,OUT-001,64
1,COV-001,15
2,DEV-001,14
3,ELG-001,11
4,EVD-001,11
5,EVD-002,11
6,DEV-002,10
7,CLM-001,9
8,DATA-002,6
9,LIM-001,5



Outcome and triggering-rule distribution:


,triage_outcome,triggering_rule_id,claim_count
0,INFO_REQUIRED,CLM-001,9
1,INFO_REQUIRED,DEV-001,14
2,INFO_REQUIRED,ELG-001,11
3,INFO_REQUIRED,EVD-001,11
4,MANUAL_REVIEW,ANM-001,4
5,MANUAL_REVIEW,DATA-002,6
6,MANUAL_REVIEW,DEV-002,10
7,MANUAL_REVIEW,EVD-002,11
8,NOT_ELIGIBLE,COV-001,15
9,NOT_ELIGIBLE,ELG-002,3



Claims routed to PROCEED:


,claim_id,triage_outcome,triggering_rule_id,precedence_stage,rule_trace_length
0,CLM-000001,PROCEED,OUT-001,6,17
1,CLM-000002,PROCEED,OUT-001,6,17
2,CLM-000003,PROCEED,OUT-001,6,17
3,CLM-000004,PROCEED,OUT-001,6,17
4,CLM-000005,PROCEED,OUT-001,6,17
5,CLM-000006,PROCEED,OUT-001,6,17
6,CLM-000007,PROCEED,OUT-001,6,17
7,CLM-000008,PROCEED,OUT-001,6,17
8,CLM-000009,PROCEED,OUT-001,6,17
9,CLM-000010,PROCEED,OUT-001,6,17



Claims with the longest audit traces:


,claim_id,triage_outcome,triggering_rule_id,precedence_stage,decision_reason,rule_trace_length,decision_support_only
0,CLM-000001,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
1,CLM-000002,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
2,CLM-000003,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
3,CLM-000004,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
4,CLM-000005,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
5,CLM-000006,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
6,CLM-000007,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
7,CLM-000008,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
8,CLM-000009,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True
9,CLM-000010,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,17,True


In [78]:
# ============================================================
# Deterministic Triage Engine — Step 25: Development Baseline Export
# ============================================================

from datetime import datetime, timezone
import json
from pathlib import Path

BASELINE_VERSION = "rules_baseline_v1"
EXPORT_DIRECTORY = Path("artifacts/baselines")

EXPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)

export_created_utc = datetime.now(timezone.utc).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

baseline_export_rows = []

for claim_id in data["development_claims"]["claim_id"]:
    context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    decision = triage_claim(context)

    baseline_export_rows.append(
        {
            "baseline_version": BASELINE_VERSION,
            "export_created_utc": export_created_utc,
            "claim_id": decision["claim_id"],
            "triage_outcome": decision["triage_outcome"],
            "triggering_rule_id": decision["triggering_rule_id"],
            "precedence_stage": decision["precedence_stage"],
            "decision_reason": decision["decision_reason"],
            "decision_support_only": decision["decision_support_only"],
            "rule_trace_length": len(decision["rule_trace"]),
            "rule_trace_json": json.dumps(
                decision["rule_trace"],
                default=str,
            ),
            "system_limitations_json": json.dumps(
                decision["system_limitations"],
                default=str,
            ),
        }
    )

baseline_export_df = pd.DataFrame(baseline_export_rows)

csv_export_path = (
    EXPORT_DIRECTORY
    / f"{BASELINE_VERSION}_development_decisions.csv"
)

jsonl_export_path = (
    EXPORT_DIRECTORY
    / f"{BASELINE_VERSION}_development_decisions.jsonl"
)

baseline_export_df.to_csv(
    csv_export_path,
    index=False,
)

with open(jsonl_export_path, "w", encoding="utf-8") as file:
    for row in baseline_export_rows:
        file.write(json.dumps(row, default=str) + "\n")

print("Development baseline export created.")
print("CSV:", csv_export_path)
print("JSONL:", jsonl_export_path)
print("Claims exported:", len(baseline_export_df))
print("Unique claim IDs:", baseline_export_df["claim_id"].nunique())
print(
    "Duplicate claim IDs:",
    baseline_export_df["claim_id"].duplicated().sum(),
)

display(
    baseline_export_df[
        [
            "claim_id",
            "triage_outcome",
            "triggering_rule_id",
            "precedence_stage",
            "rule_trace_length",
        ]
    ].head(10)
)

display(
    baseline_export_df[
        ["triage_outcome", "triggering_rule_id"]
    ]
    .value_counts()
    .rename("claim_count")
    .reset_index()
)

Development baseline export created.
CSV: artifacts/baselines/rules_baseline_v1_development_decisions.csv
JSONL: artifacts/baselines/rules_baseline_v1_development_decisions.jsonl
Claims exported: 165
Unique claim IDs: 165
Duplicate claim IDs: 0


,claim_id,triage_outcome,triggering_rule_id,precedence_stage,rule_trace_length
0,CLM-000001,PROCEED,OUT-001,6,17
1,CLM-000002,PROCEED,OUT-001,6,17
2,CLM-000003,PROCEED,OUT-001,6,17
3,CLM-000004,PROCEED,OUT-001,6,17
4,CLM-000005,PROCEED,OUT-001,6,17
5,CLM-000006,PROCEED,OUT-001,6,17
6,CLM-000007,PROCEED,OUT-001,6,17
7,CLM-000008,PROCEED,OUT-001,6,17
8,CLM-000009,PROCEED,OUT-001,6,17
9,CLM-000010,PROCEED,OUT-001,6,17


,triage_outcome,triggering_rule_id,claim_count
0,PROCEED,OUT-001,64
1,NOT_ELIGIBLE,COV-001,15
2,INFO_REQUIRED,DEV-001,14
3,INFO_REQUIRED,ELG-001,11
4,INFO_REQUIRED,EVD-001,11
5,MANUAL_REVIEW,EVD-002,11
6,MANUAL_REVIEW,DEV-002,10
7,INFO_REQUIRED,CLM-001,9
8,MANUAL_REVIEW,DATA-002,6
9,NOT_ELIGIBLE,LIM-001,5


In [79]:
# ============================================================
# Step 25A: Move Baseline Export to Project-Level Artifacts Folder
# ============================================================

from pathlib import Path
import shutil

notebook_artifact_dir = Path("artifacts/baselines")
project_artifact_dir = Path("../artifacts/baselines")

project_artifact_dir.mkdir(parents=True, exist_ok=True)

files_to_move = [
    "rules_baseline_v1_development_decisions.csv",
    "rules_baseline_v1_development_decisions.jsonl",
]

for filename in files_to_move:
    source_path = notebook_artifact_dir / filename
    target_path = project_artifact_dir / filename

    shutil.copy2(source_path, target_path)

    print("Copied:")
    print("  From:", source_path)
    print("  To:  ", target_path)

print("\nProject-level baseline artifact folder:")
print(project_artifact_dir.resolve())

Copied:
  From: artifacts/baselines/rules_baseline_v1_development_decisions.csv
  To:   ../artifacts/baselines/rules_baseline_v1_development_decisions.csv
Copied:
  From: artifacts/baselines/rules_baseline_v1_development_decisions.jsonl
  To:   ../artifacts/baselines/rules_baseline_v1_development_decisions.jsonl

Project-level baseline artifact folder:
/Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/artifacts/baselines


In [80]:
# ============================================================
# Step 25B: Verify Project-Level Baseline Export
# ============================================================

for path in project_artifact_dir.glob("rules_baseline_v1_development_decisions.*"):
    print(path.resolve())

/Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/artifacts/baselines/rules_baseline_v1_development_decisions.csv
/Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/artifacts/baselines/rules_baseline_v1_development_decisions.jsonl


In [81]:
# ============================================================
# Step 25C: Create Baseline Manifest
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json

BASELINE_VERSION = "rules_baseline_v1"
PROJECT_BASELINE_DIR = Path("../artifacts/baselines")

csv_path = (
    PROJECT_BASELINE_DIR
    / "rules_baseline_v1_development_decisions.csv"
)

jsonl_path = (
    PROJECT_BASELINE_DIR
    / "rules_baseline_v1_development_decisions.jsonl"
)

def file_sha256(path: Path) -> str:
    """Return the SHA-256 checksum for a file."""
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            digest.update(chunk)

    return digest.hexdigest()


manifest = {
    "baseline_version": BASELINE_VERSION,
    "artifact_type": "deterministic_rules_baseline",
    "created_utc": datetime.now(timezone.utc).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    ),
    "source_dataset": "development_claims",
    "claim_count": int(len(baseline_export_df)),
    "unique_claim_count": int(
        baseline_export_df["claim_id"].nunique()
    ),
    "outcome_distribution": (
        baseline_export_df["triage_outcome"]
        .value_counts()
        .to_dict()
    ),
    "triggering_rule_distribution": (
        baseline_export_df["triggering_rule_id"]
        .value_counts()
        .to_dict()
    ),
    "system_limitations": [
        (
            "DEV-003 is not evaluated because the runtime context does "
            "not provide a verified NOT_ENROLLED device decision."
        ),
        (
            "EXC-001 and EXC-002 are not evaluated because no structured "
            "exclusion-status dataset is available."
        ),
        (
            "Claim-history source completeness cannot be independently "
            "verified from the current runtime package."
        ),
    ],
    "artifacts": {
        "csv": {
            "filename": csv_path.name,
            "sha256": file_sha256(csv_path),
        },
        "jsonl": {
            "filename": jsonl_path.name,
            "sha256": file_sha256(jsonl_path),
        },
    },
}

manifest_path = (
    PROJECT_BASELINE_DIR
    / "rules_baseline_v1_manifest.json"
)

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print("Baseline manifest created:")
print(manifest_path.resolve())

print("\nManifest summary:")
print("Claims:", manifest["claim_count"])
print("Unique claims:", manifest["unique_claim_count"])
print("CSV checksum:", manifest["artifacts"]["csv"]["sha256"][:16] + "...")
print("JSONL checksum:", manifest["artifacts"]["jsonl"]["sha256"][:16] + "...")

Baseline manifest created:
/Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/artifacts/baselines/rules_baseline_v1_manifest.json

Manifest summary:
Claims: 165
Unique claims: 165
CSV checksum: 87e96250624d9271...
JSONL checksum: 0cb6f513713e4d40...


In [82]:
# ============================================================
# Step 26: Deterministic Triage Tool Validation
# ============================================================

from src.tools.deterministic_triage import run_deterministic_triage

tool_validation_claims = {
    "Clean claim": "CLM-000001",
    "Risk-triggered claim": "CLM-000180",
    "Missing evidence": "CLM-000117",
    "Annual limit exhausted": "CLM-000204",
}

tool_validation_rows = []

for scenario, claim_id in tool_validation_claims.items():
    tool_result = run_deterministic_triage(
        data=data,
        claim_id=claim_id,
    )

    direct_context = claim_context_module.assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    direct_decision = triage_claim(direct_context)
    tool_decision = tool_result["deterministic_decision"]

    assert tool_decision == direct_decision

    tool_validation_rows.append(
        {
            "scenario": scenario,
            "claim_id": claim_id,
            "tool_name": tool_result["tool_name"],
            "tool_version": tool_result["tool_version"],
            "triage_outcome": tool_decision["triage_outcome"],
            "triggering_rule_id": tool_decision[
                "triggering_rule_id"
            ],
            "matches_direct_engine": tool_decision == direct_decision,
        }
    )

tool_validation_df = pd.DataFrame(tool_validation_rows)

display(tool_validation_df)

,scenario,claim_id,tool_name,tool_version,triage_outcome,triggering_rule_id,matches_direct_engine
0,Clean claim,CLM-000001,deterministic_triage,rules_baseline_v1,PROCEED,OUT-001,True
1,Risk-triggered claim,CLM-000180,deterministic_triage,rules_baseline_v1,MANUAL_REVIEW,ANM-001,True
2,Missing evidence,CLM-000117,deterministic_triage,rules_baseline_v1,INFO_REQUIRED,EVD-001,True
3,Annual limit exhausted,CLM-000204,deterministic_triage,rules_baseline_v1,NOT_ELIGIBLE,LIM-001,True
